## 0. Setup

Imports, working directory, and GEE initialisation.

In [2]:
import pandas as pd
import geopandas as gpd
import ee
import geemap
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(override=True)

os.chdir("/Users/esthergreenwood/Documents/GitHub/mapping_sanitation_access_LMICs")
print("cwd:", Path.cwd())


/Users/esthergreenwood/Documents/GitHub/mapping_sanitation_access_LMICs/.venv/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/esthergreenwood/Documents/GitHub/mapping_sanitation_access_LMICs/.venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/esthergreenwood/Documents/GitHub/mapping_sanitation_access_LMICs/.venv/lib/python3.9/site-p

cwd: /Users/esthergreenwood/Documents/GitHub/mapping_sanitation_access_LMICs


In [ ]:

#ee.Authenticate(auth_mode='localhost') # one time browser login

In [3]:

ee.Initialize(project=os.getenv("GEE_USERNAME")) # intitialize every session

## 1. Load Boundary Data

Loads DHS, MICS, other-survey, and GADM shapefiles from local data directories.

In [4]:

# DHS_boundary data are in directory with per country subdirectories containing the DHS region shapefiles (in subdir 'shps')
# MICS_boundary data are in directory with per country subdirectories containing the MICS region shapefiles (in subdir 'Shapefiles'); one shapefile per admin level
# other_boundary data includes one shapefile for Pakistan with country subdirectories at level 2 (in subdir 'Shapefiles' name: other_Pakistan_boundaries2.shp)
# and one shapefile for Nepal at level 1 (in subdir 'Shapefiles' name: national_Nepal_boundaries1.shp); 
# GADM_boundary data includes one shapefile with 130 low and middle income country (World Bank income groups 2024) subdirectories at level 1

DHS_boundaries = 'data/DHS_boundaries'
MICS_boundaries = 'data/MICS_boundaries'
other_boundaries = 'data/other_boundaries'
GADM_lmic_shp = 'data/GADM36_boundaries/gadm_lmics_final.shp'

# Load all DHS shapefiles, preferring level 2 if available
dhs_gdfs = []

for country_dir in Path(DHS_boundaries).iterdir():
    if not country_dir.is_dir():
        continue
    shps_dir = country_dir / 'shps'
    shp_level2 = shps_dir / 'sdr_subnational_boundaries2.shp'
    shp_level1 = shps_dir / 'sdr_subnational_boundaries.shp'
    if shp_level2.exists():
        shp = shp_level2
        admin_level = 2
    elif shp_level1.exists():
        shp = shp_level1
        admin_level = 1
    else:
        continue
    gdf = gpd.read_file(shp)
    gdf["admin_level"] = admin_level
    gdf["dhs_boundary_file"] = shp.name
    dhs_gdfs.append(gdf)

dhs_gdf = gpd.GeoDataFrame(pd.concat(dhs_gdfs, ignore_index=True))

# Load MICS water-quality shapefiles from Countries_Water_Quality
# Prefer country-specific files: mics_boundaries_<country>.shp

mics_gdfs = []
mics_wq_dir = Path(MICS_boundaries) / "Countries_Water_Quality"

for country_dir in sorted(mics_wq_dir.iterdir()):
    if not country_dir.is_dir():
        continue

    shapefiles_dir = country_dir / "Shapefiles"
    if not shapefiles_dir.exists():
        print(f"Missing Shapefiles folder: {country_dir}")
        continue

    folder_first_letter = country_dir.name[0].lower()

    matching_shps = [
        shp for shp in sorted(shapefiles_dir.glob("mics_boundaries_*.shp"))
        if shp.stem.replace("mics_boundaries_", "")[:1].lower() == folder_first_letter
    ]

    if not matching_shps:
        print(f"No country-specific MICS shapefile found for {country_dir.name}")
        continue

    if len(matching_shps) > 1:
        print(f"Multiple matching MICS shapefiles for {country_dir.name}: {[p.name for p in matching_shps]}")
        print(f"Using: {matching_shps[0].name}")

    shp = matching_shps[0]

    gdf = gpd.read_file(shp)

    if "LEVELCO" not in gdf.columns:
        print(f"Warning: LEVELCO missing in {shp}")
        gdf["admin_level"] = None
    else:
        gdf["admin_level"] = gdf["LEVELCO"]

    gdf["mics_folder"] = country_dir.name
    gdf["mics_boundary_file"] = shp.name

    mics_gdfs.append(gdf)

    mics_gdf = gpd.GeoDataFrame(
        pd.concat([gdf for gdf in mics_gdfs if not gdf.empty], ignore_index=True)
    )

print("Loaded MICS water-quality boundary files:")
print(
    mics_gdf[["mics_folder", "mics_boundary_file", "admin_level"]]
    .drop_duplicates()
    .to_string(index=False)
)

# Load MICS "other" shapefiles from Countries_other
# Prefer country-specific files: mics_boundaries_<country>.shp

mics_other_gdfs = []
mics_other_dir = Path(MICS_boundaries) / "Countries_other"

for country_dir in sorted(mics_other_dir.iterdir()):
    if not country_dir.is_dir():
        continue

    shapefiles_dir = country_dir / "Shapefiles"
    if not shapefiles_dir.exists():
        print(f"Missing Shapefiles folder: {country_dir}")
        continue

    folder_first_letter = country_dir.name[0].lower()

    matching_shps = [
        shp for shp in sorted(shapefiles_dir.glob("mics_boundaries_*.shp"))
        if shp.stem.replace("mics_boundaries_", "")[:1].lower() == folder_first_letter
    ]

    if not matching_shps:
        print(f"No country-specific MICS shapefile found for {country_dir.name}")
        continue

    if len(matching_shps) > 1:
        print(f"Multiple matching MICS shapefiles for {country_dir.name}: {[p.name for p in matching_shps]}")
        print(f"Using: {matching_shps[0].name}")

    shp = matching_shps[0]
    gdf = gpd.read_file(shp)

    gdf["admin_level"] = gdf["LEVELCO"] if "LEVELCO" in gdf.columns else None
    gdf["mics_folder"] = country_dir.name
    gdf["mics_boundary_file"] = shp.name
    gdf["mics_type"] = "other"

    mics_other_gdfs.append(gdf)

    mics_other_gdf = gpd.GeoDataFrame(
        pd.concat([gdf for gdf in mics_other_gdfs if not gdf.empty], ignore_index=True)
    )


# Load other boundary shapefiles: Pakistan level 2, Nepal level 1, Kazakhstan level 1

other_gdfs = []
shapefiles_dir = Path(other_boundaries) / "Shapefiles"

other_files = [
    {
        "filename": "WFP_Pakistan_boundaries2.shp",
        "country": "Pakistan",
        "admin_level": 2,
        "analysis_year": 2018,
        "source": "other"
    },
    {
        "filename": "national_Nepal_boundaries1.shp",
        "country": "Nepal",
        "admin_level": 1,
        "analysis_year": 2024,
        "source": "other"
    },
    {
        "filename": "kaz_admbnda_adm1_unhcr_2023.shp",
        "country": "Kazakhstan",
        "admin_level": 1,
        "analysis_year": 2024,
        "source": "other"
    }
]

if shapefiles_dir.exists():
    for item in other_files:
        shp = shapefiles_dir / item["filename"]

        if shp.exists():
            gdf = gpd.read_file(shp)

            print("\nLoaded:", shp.name)
            print("Country assigned:", item["country"])
            print("Original CRS:", gdf.crs)
            print("Columns:", list(gdf.columns))
            print("Rows:", len(gdf))

            # Force all other boundaries to WGS84 before combining
            if gdf.crs is None:
                raise ValueError(f"CRS missing for {shp.name}")
            
            gdf = gdf.to_crs("EPSG:4326")

            # Standardise Kazakhstan region-name column for later code
            if item["country"] == "Kazakhstan" and "ADM1_EN" in gdf.columns:
                gdf["adm1_name"] = gdf["ADM1_EN"]

            gdf["country"] = item["country"]
            gdf["admin_level"] = item["admin_level"]
            gdf["analysis_year"] = item["analysis_year"]
            gdf["source"] = item["source"]

            print("Converted CRS:", gdf.crs)

            other_gdfs.append(gdf)

        else:
            print(f"Missing file: {shp}")

    if len(other_gdfs) == 0:
        raise ValueError("No other boundary shapefiles were loaded.")

    other_gdf = gpd.GeoDataFrame(
        pd.concat(other_gdfs, ignore_index=True),
        crs="EPSG:4326"
    )

else:
    raise FileNotFoundError(f"Folder not found: {shapefiles_dir}")

# Load merged GADM shapefile

gadm_lmic_gdf = gpd.read_file(GADM_lmic_shp)

# Drop null/empty geometries and force 2D
dhs_gdf = dhs_gdf[~(dhs_gdf.geometry.isna() | dhs_gdf.geometry.is_empty)].copy()
dhs_gdf['geometry'] = dhs_gdf['geometry'].force_2d()
mics_gdf = mics_gdf[~(mics_gdf.geometry.isna() | mics_gdf.geometry.is_empty)].copy()
mics_gdf['geometry'] = mics_gdf['geometry'].force_2d()
mics_other_gdf = mics_other_gdf[~(mics_other_gdf.geometry.isna() | mics_other_gdf.geometry.is_empty)].copy()
mics_other_gdf["geometry"] = mics_other_gdf.geometry.force_2d()
other_gdf = other_gdf[~(other_gdf.geometry.isna() | other_gdf.geometry.is_empty)].copy()
other_gdf['geometry'] = other_gdf['geometry'].force_2d()
gadm_lmic_gdf = gadm_lmic_gdf[~(gadm_lmic_gdf.geometry.isna() | gadm_lmic_gdf.geometry.is_empty)].copy()
gadm_lmic_gdf['geometry'] = gadm_lmic_gdf.geometry.force_2d()

# Add analysis year column to both dataframes (DHS uses 'SVYYEAR', MICS uses 'SVYYEARS')
dhs_gdf['analysis_year'] = dhs_gdf['SVYYEAR'].astype(int)
##dhs_gdf['admin_level'] = 1  # DHS always one level per survey
mics_gdf['analysis_year'] = mics_gdf['SVYYEARS']
mics_gdf["source"] = "MICS"

mics_other_gdf["analysis_year"] = mics_other_gdf["SVYYEARS"]
mics_other_gdf["source"] = "MICS_other"

gadm_lmic_gdf['analysis_year'] = 2024

gadm_lmic_gdf['admin_level'] = 1
gadm_lmic_gdf['source'] = 'GADM'

print('DHS size:', len(dhs_gdf))
print('MICS size:', len(mics_gdf))
print('other size:', len(other_gdf))
print('GADM size:', len(gadm_lmic_gdf))

print("DHS CRS:", dhs_gdf.crs)
print("MICS CRS:", mics_gdf.crs)
print("Other CRS:", other_gdf.crs)
print("GADM CRS:", gadm_lmic_gdf.crs)

# Print countries included from DHS and MICS
print("\nDHS countries included:")
if "CNTRYNAMEE" in dhs_gdf.columns:
    dhs_countries = sorted(dhs_gdf["CNTRYNAMEE"].dropna().unique())
    for country in dhs_countries:
        print(country)
    print("Total DHS countries:", len(dhs_countries))
else:
    print("Column CNTRYNAMEE not found in dhs_gdf")

print("\nMICS countries included with water quality:")
if "CNTRYNAME" in mics_gdf.columns:
    mics_countries = sorted(mics_gdf["CNTRYNAME"].dropna().unique())
    for country in mics_countries:
        print(country)
    print("Total MICS countries:", len(mics_countries))
else:
    print("Column CNTRYNAME not found in mics_gdf")

print("\nOther MICS countries included without water quality:")
if "CNTRYNAME" in mics_other_gdf.columns:
    mics_countries_other = sorted(mics_other_gdf["CNTRYNAME"].dropna().unique())
    for country in mics_countries_other:
        print(country)
    print("Total MICS countries:", len(mics_countries_other))
else:
    print("Column CNTRYNAME not found in mics_other_gdf")


/Users/esthergreenwood/Documents/GitHub/mapping_sanitation_access_LMICs/.venv/lib/python3.9/site-packages/pyogrio/raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Polygon' is converted to 'Polygon Z'
  return ogr_read(
/Users/esthergreenwood/Documents/GitHub/mapping_sanitation_access_LMICs/.venv/lib/python3.9/site-packages/pyogrio/raw.py:198: RuntimeWarning: data/MICS_boundaries/Countries_Water_Quality/GambiaMICS2018Boundaries_2/Shapefiles/mics_boundaries_gambia.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


Loaded MICS water-quality boundary files:
                                      mics_folder              mics_boundary_file      admin_level
                              EswatiniMICS2021-22    mics_boundaries_eswatini.shp          Admin 1
                       GambiaMICS2018Boundaries_2      mics_boundaries_gambia.shp          Admin 1
                      GeorgiaMICS2018Boundaries_2     mics_boundaries_georgia.shp          Admin 1
                      GeorgiaMICS2018Boundaries_2     mics_boundaries_georgia.shp Group of Admin 1
                      GeorgiaMICS2018Boundaries_2     mics_boundaries_georgia.shp             None
                       GhanaMICS2017-18Boundaries       mics_boundaries_ghana.shp          Admin 1
                       HondurasMICS2019Boundaries    mics_boundaries_honduras.shp          Admin 1
                       HondurasMICS2019Boundaries    mics_boundaries_honduras.shp             None
         Kosovo(UNSCR1244)MICS2019-20Boundaries_3      mics_boundar

### 1a. Water Quality Survey Configuration

Defines admin-area geometries and manually defines analysis-year for the water quality dataset where no shapefiles were available.

In [6]:
# Defining adminstrative areas reported in household surveys
# Define island groups for Kiribati
noGil = ee.Geometry.Polygon(
        [[[173.76214858993802, 1.688437284048748],
          [173.89398452743802, 3.2689380495714633],
          [172.53167983993802, 4.277476168442425],
          [172.05246629783107, 3.029716351603881],
          [172.84548591405644, 1.4656846206052414],
          [173.12175664737128, 1.3820186787491144],
          [173.1354466421833, 1.3865663776048271],
          [173.14021024539375, 1.3954043333854713]]], None, False)
souTara = ee.Geometry.Polygon(
    [[[173.14024407183683, 1.3954382342837202],
        [173.13552338397062, 1.3865573758520389],
        [173.12591034686125, 1.382867734037598],
        [172.9929535267801, 1.3664672516799288],
        [172.8940765736551, 1.3946115454160364],
        [172.8940765736551, 1.317042332174653],
        [173.04582523088166, 1.304685947700206],
        [173.21679996232697, 1.3609756435068756],
        [173.17560123185822, 1.406967470897869]]], None, False)
cenGil = ee.Geometry.Polygon(
    [[[172.75257236267254, 1.106559466163889],
        [172.75257236267254, -0.947771870448704],
        [174.70813876892254, -0.947771870448704],
        [174.70813876892254, 1.106559466163889]]], None, False)
souGil = ee.Geometry.Polygon(
    [[[174.44446689392254, -1.0356493216635867],
        [174.44446689392254, -2.858249945133194],
        [177.15808994079754, -2.858249945133194],
        [177.15808994079754, -1.0356493216635867]]], None, False)
lineAndPhoenix =  ee.Geometry.Polygon(
    [[[178.7565749612447, 8.140225397293124],
        [178.7565749612447, -20.88502241263751],
        [223.75657496124472, -20.88502241263751],
        [223.75657496124472, 8.140225397293124]]], None, False)

# Define the countries to sample from level 0, 2, 3 or 1
countryLevel0 = ['Kiribati']
countryLevel1 = ['Gambia','Iraq', 'Chad','Georgia','Ghana','Guinea-Bissau','Guyana','Laos','Lesotho','Nigeria','Suriname','Tonga','Zimbabwe','Democratic Republic of the Congo','Benin','Honduras','Tuvalu','Vanuatu','São Tomé and Príncipe']
countryLevel2 = ['Gambia','Kosovo','Madagascar','Palestina']
countryLevel3 = ['Pakistan']

# Load the GADM datasets 

GADM_level0 = ee.FeatureCollection('users/crowtherlab/GADM36/GADM36_level0') \
                    .filter(ee.Filter.inList('NAME_0',['Kiribati']))

GADM_level1_prep = ee.FeatureCollection('users/crowtherlab/GADM36/GADM36_level1')

GADM_level1 = GADM_level1_prep.filter(ee.Filter.inList('NAME_0',countryLevel1)) \
                                  .map(lambda f: f.set('HH7',f.getString('NAME_1')))

GADM_level2_prep = ee.FeatureCollection('users/crowtherlab/GADM36/GADM36_level2')
GADM_level2 = GADM_level2_prep.filter(ee.Filter.inList('NAME_0',countryLevel2)) \
                                  .map(lambda f: f.set('HH7',f.getString('NAME_2')))
                    
GADM_level3_prep = ee.FeatureCollection('users/crowtherlab/GADM36/GADM36_level3')
GADM_level3 = GADM_level3_prep \
                    .filter(ee.Filter.inList('NAME_0',countryLevel3)) \
                    .map(lambda f: f.set('HH7',f.getString('NAME_3')))


##/ Define the regions for Level 0
# Kiribati, define using geometry (see above)
# Define Kiribati 
NORTHERN_GILBERT = ee.Feature(GADM_level0.geometry().intersection(noGil, 10)).set('NAME_1','NORTHERN GILBERT').set('NAME_0','Kiribati').set('HH7','NORTHERN_GILBERT')
SOUTH_TARAWA = ee.Feature(GADM_level0.geometry().intersection(souTara, 10)).set('NAME_1','SOUTH TARAWA').set('NAME_0','Kiribati').set('HH7','SOUTH_TARAWA')
CENTRAL_GILBERT = ee.Feature(GADM_level0.geometry().intersection(cenGil, 10)).set('NAME_1','CENTRAL GILBERT').set('NAME_0','Kiribati').set('HH7','CENTRAL_GILBERT')
SOUTHERN_GILBERT = ee.Feature(GADM_level0.geometry().intersection(souGil, 10)).set('NAME_1','SOUTHERN GILBERT').set('NAME_0','Kiribati').set('HH7','SOUTHERN_GILBERT')
LINE_AND_PHOENIX = ee.Feature(GADM_level0.geometry().intersection(lineAndPhoenix, 10)).set('NAME_1','LINE AND PHOENIX GROUP').set('NAME_0','Kiribati').set('HH7','LINE_AND_PHOENIX')
GADM_level0 = ee.FeatureCollection([NORTHERN_GILBERT, SOUTH_TARAWA, CENTRAL_GILBERT, SOUTHERN_GILBERT, LINE_AND_PHOENIX])


##/ Define the regions for Level 1
# Algeria 
NORD_CENTRE = ['Alger','Blida','Boumerdès','Tipaza','Bouira','Médéa','Tizi Ouzou','Béjaïa','Chlef','Aïn Defla']
NORD_EST = ['Annaba','Constantine','Skikda','Jijel','Mila','Souk Ahras','El Tarf','Guelma']
NORD_OUEST = ['Oran','Tlemcen','Mostaganem','Aïn Témouchent','Relizane','Sidi Bel Abbès','Mascara']
HAUT_PLATEAU_CENTRE = ['Djelfa','Laghouat',"M'Sila"]
HAUT_PLATEAU_EST = ['Sétif','Batna','Khenchela','Bordj Bou Arréridj','Oum el Bouaghi','Tébessa']
HAUT_PLATEAU_OUEST = ['Saïda','Tissemsilt','Naâma','El Bayadh','Tiaret']
SUD = ['Béchar','Tindouf','Adrar','Ghardaïa','Biskra','El Oued','Ouargla','Tamanghasset','Illizi']

# Azerbaijan (New)
BAKU = ['Bakı']
NAKHCHIVAN_AR = ['Nakhchivan']
ABSHERON_KHIZI = ['Xizı', 'Sumqayıt', 'Abşeron']
DAGHLIG_SHIRVAN = ['Daglig-Shirvan']
GANJA_DASHKASAN = ['Samux', 'Gəncə', 'Goranboy', 'Xanlar', 'Daşkəsən', 'Naftalan']
KARABAKH_L1 = ['Yukhari-Karabakh']
KARABAKH_L2 = ['Bərdə', 'Ağcabədi']
GAZAKH_TOVUZ = ['Ağstafa', 'Qazax', 'Tovuz', 'Şəmkir', 'Gədəbəy']
GUBA_KHACHMAZ = ['Quba-Khachmaz']
LANKARAN_ASTARA = ['Lankaran']
CENTRAL_ARAN = ['Mingəçevir', 'Yevlax', 'Yevlax City', 'Ağdaş', 'Göyçay', 'Ucar', 'Kürdəmir', 'Zərdab']
MIL_MUGHAN = ['Sabirabad', 'Saatlı', 'İmişli', 'Beyləqan']
SHAKI_ZAGATALA = ['Shaki-Zaqatala']
SHIRVAN_SALYAN = ['Hacıqabul', 'Əli Bayramlı', 'Salyan', 'Biləsuvar', 'Neftçala']

# Bangladesh
Barishal = ["Barisal"]
Chattogram = ["Chittagong"]
Dhaka = ["Dhaka"]
Khulna = ["Khulna"]
Mymenshing = ["Jamalpur", "Mymensingh", "Netrakona", "Sherpur"]
Rajshahi = ["Rajshahi"]
Rangpur = ["Rangpur"]
Sylhet = ["Sylhet"]

# Central African Republic 
REGION_1 = ["Ombella-M'Poko","Lobaye"]
REGION_2 = ["Mambéré-Kadéï","Nana-Mambéré","Sangha-Mbaéré"]
REGION_3 = ["Ouham-Pendé","Ouham"]
REGION_4 = ["Kémo","Nana-Grébizi","Ouaka"]
REGION_5 = ["Bamingui-Bangoran","Haute-Kotto","Vakaga"]
REGION_6 = ["Basse-Kotto","Mbomou","Haut-Mbomou"]
REGION_7 = ["Bangui"]

# Dominican Republic (New)
OZAMA = ['Distrito Nacional','Santo Domingo']
VALDESIA = ['Peravia','San José de Ocoa','San Cristóbal']
ENRIQUILLO = ['Barahona','Bahoruco','Independencia','Pedernales']
CIBAO_NORDESTE = ['Duarte','Salcedo','María Trinidad Sánchez','Samaná']
CIBAO_NOROESTE = ['Dajabón','Monte Cristi','Santiago Rodríguez','Valverde']
CIBAO_NORTE = ['Espaillat','Puerto Plata','Santiago']
CIBAO_SUR = ['La Vega','Monseñor Nouel','Sánchez Ramírez']
EL_VALLE = ['La Estrelleta','San Juan','Azua']
HIGUAMO = ['Hato Mayor','Monte Plata','San Pedro de Macorís']
YUMA = ['El Seybo','La Romana','La Altagracia']

# Mongolia
WESTERN = ["Bayan-Ölgiy","Govi-Altay","Dzavhan","Hovd","Uvs"]
EASTERN = ["Dornod","Sühbaatar","Hentiy"]
ULAANBAATAR = ["Ulaanbaatar"]
CENTRAL = ["Töv","Ömnögovi","Selenge","Orhon","Darhan-Uul","Govisümber","Dundgovi","Dornogovi"]
KHANGAI = ["Bayanhongor","Hövsgöl","Arhangay","Övörhangay","Bulgan"]


# Paraguay
Asuncion = ["Asunción"]
Resto = [
    "Amambay",
    "Caazapá",
    "Canindeyú",
    "Concepción",
    "Cordillera",
    "Guairá",
    "Misiones",
    "Ñeembucú",
    "Paraguarí",
    "Presidente Hayes"
]
San_Pedro = ["San Pedro"]
Caaguazu = ["Caaguazú"]
Itapua = ["Itapúa"]
Alto_Parana = ["Alto Paraná"]
Central = ["Central"]
Alto_Paraguay = ["Alto Paraguay"]
Boqueron = ["Boquerón"]


# Sierra Leone

EAST = ["Kailahun", "Kenema", "Kono"]

NORTH = ["Koinadugu", "Bombali", "Kambia", "Port Loko", "Tonkolili"]

SOUTH = ["Bo", "Bonthe", "Moyamba", "Pujehun"]

WEST = ["Western Area Rural", "Western Area Urban"]


# Togo
Maritime = ["Maritime"]
Plateaux = ["Plateaux"]
Centrale = ["Centre"]
Kara = ["Kara"]
Savanes = ["Savanes"]
Lomé_Commune_Golfe_Urbain = ["Golfe (incl Lomé)"]

# Tunisia (new)
TUNISIA_DISTRICT_TUNIS = ['Tunis', 'Ariana', 'Ben Arous (Tunis Sud)', 'Manubah'] #corrected
TUNISIA_NORD_EST = ['Bizerte','Nabeul','Zaghouan']
TUNISIA_NORD_OUEST = ['Béja','Jendouba','Le Kef','Siliana']
TUNISIA_CENTRE_EST = ['Mahdia','Monastir','Sfax','Sousse']
TUNISIA_CENTRE_OUEST = ['Kassérine','Kairouan','Sidi Bou Zid']
TUNISIA_SUD_EST = ['Gabès','Médenine','Tataouine']
TUNISIA_SUD_OUEST = ['Gafsa','Kebili','Tozeur']

# Samoa
UPOLU = ["A'ana", "Aiga-i-le-Tai", "Atua", "Tuamasaga", "Va'a-o-Fonoti"]
SAVAII = ["Fa'asaleleaga", "Gaga'emauga", "Gagaifomauga", "Palauli", "Satupa'itea", "Vaisigano"]

# Vietnam (New)
NORTHERN_MIDLANDS_AND_MOUNTAIN = ['Hà Giang','Cao Bằng','Lào Cai','Lai Châu','Bắc Kạn','Tuyên Quang','Lạng Sơn','Yên Bái','Điện Biên','Thái Nguyên','Bắc Giang','Phú Thọ','Sơn La','Hòa Bình'] #corrected
RED_RIVER_DELTA = ['Vĩnh Phúc','Quảng Ninh','Bắc Ninh','Hà Nội','Hải Dương','Hưng Yên','Hải Phòng','Hà Nam','Thái Bình','Nam Định','Ninh Bình']
NORTH_CENTRAL_AND_COASTAL = ['Thanh Hóa','Nghệ An','Hà Tĩnh','Quảng Bình','Quảng Trị','Thừa Thiên Huế','Đà Nẵng','Quảng Nam','Quảng Ngãi','Bình Định','Phú Yên','Khánh Hòa','Ninh Thuận','Bình Thuận']
CENTRAL_HIGHLANDS = ['Kon Tum','Gia Lai','Đắk Lắk','Đăk Nông','Lâm Đồng']
SOUTH_EAST = ['Bình Phước','Tây Ninh','Bình Dương','Đồng Nai','Hồ Chí Minh','Bà Rịa - Vũng Tàu']
MEKONG_RIVER_DELTA = ['Long An','Đồng Tháp','An Giang','Tiền Giang','Bến Tre','Cần Thơ','Vĩnh Long','Kiên Giang','Trà Vinh','Hậu Giang','Sóc Trăng','Bạc Liêu','Cà Mau'] #corrected

# Concatenate the districts into regions
#Algeria
NORD_CENTRE = GADM_level1_prep.filterMetadata('NAME_0','equals','Algeria').filter(ee.Filter.inList("NAME_1",NORD_CENTRE))
NORD_EST = GADM_level1_prep.filterMetadata('NAME_0','equals','Algeria').filter(ee.Filter.inList("NAME_1",NORD_EST))
NORD_OUEST = GADM_level1_prep.filterMetadata('NAME_0','equals','Algeria').filter(ee.Filter.inList("NAME_1",NORD_OUEST))
HAUT_PLATEAU_CENTRE = GADM_level1_prep.filterMetadata('NAME_0','equals','Algeria').filter(ee.Filter.inList("NAME_1",HAUT_PLATEAU_CENTRE))
HAUT_PLATEAU_EST = GADM_level1_prep.filterMetadata('NAME_0','equals','Algeria').filter(ee.Filter.inList("NAME_1",HAUT_PLATEAU_EST))
HAUT_PLATEAU_OUEST = GADM_level1_prep.filterMetadata('NAME_0','equals','Algeria').filter(ee.Filter.inList("NAME_1",HAUT_PLATEAU_OUEST))
SUD = GADM_level1_prep.filterMetadata('NAME_0','equals','Algeria').filter(ee.Filter.inList("NAME_1",SUD))

# Azerbaijan (new)
BAKU = GADM_level2_prep.filterMetadata('NAME_0','equals','Azerbaijan').filter(ee.Filter.inList("NAME_2", BAKU))
NAKHCHIVAN_AR = GADM_level1_prep.filterMetadata('NAME_0','equals','Azerbaijan').filter(ee.Filter.inList("NAME_1", NAKHCHIVAN_AR))
ABSHERON_KHIZI = GADM_level2_prep.filterMetadata('NAME_0','equals','Azerbaijan').filter(ee.Filter.inList("NAME_2", ABSHERON_KHIZI))
DAGHLIG_SHIRVAN = GADM_level1_prep.filterMetadata('NAME_0','equals','Azerbaijan').filter(ee.Filter.inList("NAME_1", DAGHLIG_SHIRVAN))
GANJA_DASHKASAN = GADM_level2_prep.filterMetadata('NAME_0','equals','Azerbaijan').filter(ee.Filter.inList("NAME_2", GANJA_DASHKASAN))
KARABAKH = (
    GADM_level1_prep.filterMetadata('NAME_0','equals','Azerbaijan').filter(ee.Filter.inList("NAME_1", KARABAKH_L1))
    .merge(GADM_level2_prep.filterMetadata('NAME_0','equals','Azerbaijan').filter(ee.Filter.inList("NAME_2", KARABAKH_L2)))
)
GAZAKH_TOVUZ = GADM_level2_prep.filterMetadata('NAME_0','equals','Azerbaijan').filter(ee.Filter.inList("NAME_2", GAZAKH_TOVUZ))
GUBA_KHACHMAZ = GADM_level1_prep.filterMetadata('NAME_0','equals','Azerbaijan').filter(ee.Filter.inList("NAME_1", GUBA_KHACHMAZ))
LANKARAN_ASTARA = GADM_level1_prep.filterMetadata('NAME_0','equals','Azerbaijan').filter(ee.Filter.inList("NAME_1", LANKARAN_ASTARA))
CENTRAL_ARAN = GADM_level2_prep.filterMetadata('NAME_0','equals','Azerbaijan').filter(ee.Filter.inList("NAME_2", CENTRAL_ARAN))
MIL_MUGHAN = GADM_level2_prep.filterMetadata('NAME_0','equals','Azerbaijan').filter(ee.Filter.inList("NAME_2", MIL_MUGHAN))
SHAKI_ZAGATALA = GADM_level1_prep.filterMetadata('NAME_0','equals','Azerbaijan').filter(ee.Filter.inList("NAME_1", SHAKI_ZAGATALA))
SHIRVAN_SALYAN = GADM_level2_prep.filterMetadata('NAME_0','equals','Azerbaijan').filter(ee.Filter.inList("NAME_2", SHIRVAN_SALYAN))

# Bangladesh
Barishal = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Bangladesh").filter(ee.Filter.inList("NAME_1", Barishal))
Chattogram = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Bangladesh").filter(ee.Filter.inList("NAME_1", Chattogram))
Dhaka_full = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Bangladesh").filter(ee.Filter.inList("NAME_1", Dhaka))
Khulna = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Bangladesh").filter(ee.Filter.inList("NAME_1", Khulna))
Rajshahi = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Bangladesh").filter(ee.Filter.inList("NAME_1", Rajshahi))
Rangpur = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Bangladesh").filter(ee.Filter.inList("NAME_1", Rangpur))
Sylhet = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Bangladesh").filter(ee.Filter.inList("NAME_1", Sylhet))

Mymenshing = GADM_level2_prep.filterMetadata("NAME_0", "equals", "Bangladesh").filter(ee.Filter.inList("NAME_2", Mymenshing))

Dhaka = ee.FeatureCollection([
    ee.Feature(Dhaka_full.geometry().difference(Mymenshing.geometry(), 10))
    .set("NAME_0", "Bangladesh")
    .set("NAME_1", "Dhaka")
    .set("HH7", "Dhaka")
])


# Central African Republic 
REGION_1 = GADM_level1_prep.filterMetadata('NAME_0','equals','Central African Republic').filter(ee.Filter.inList("NAME_1",REGION_1))
REGION_2 = GADM_level1_prep.filterMetadata('NAME_0','equals','Central African Republic').filter(ee.Filter.inList("NAME_1",REGION_2))
REGION_3 = GADM_level1_prep.filterMetadata('NAME_0','equals','Central African Republic').filter(ee.Filter.inList("NAME_1",REGION_3))
REGION_4 = GADM_level1_prep.filterMetadata('NAME_0','equals','Central African Republic').filter(ee.Filter.inList("NAME_1",REGION_4))
REGION_5 = GADM_level1_prep.filterMetadata('NAME_0','equals','Central African Republic').filter(ee.Filter.inList("NAME_1",REGION_5))
REGION_6 = GADM_level1_prep.filterMetadata('NAME_0','equals','Central African Republic').filter(ee.Filter.inList("NAME_1",REGION_6))
REGION_7 = GADM_level1_prep.filterMetadata('NAME_0','equals','Central African Republic').filter(ee.Filter.inList("NAME_1",REGION_7))

# Mongolia
WESTERN = GADM_level1_prep.filterMetadata('NAME_0','equals','Mongolia').filter(ee.Filter.inList("NAME_1",WESTERN))
EASTERN = GADM_level1_prep.filterMetadata('NAME_0','equals','Mongolia').filter(ee.Filter.inList("NAME_1",EASTERN))
ULAANBAATAR = GADM_level1_prep.filterMetadata('NAME_0','equals','Mongolia').filter(ee.Filter.inList("NAME_1",ULAANBAATAR))
CENTRAL = GADM_level1_prep.filterMetadata('NAME_0','equals','Mongolia').filter(ee.Filter.inList("NAME_1",CENTRAL))
KHANGAI = GADM_level1_prep.filterMetadata('NAME_0','equals','Mongolia').filter(ee.Filter.inList("NAME_1",KHANGAI))

# Paraguay
Asuncion = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Paraguay").filter(ee.Filter.inList("NAME_1", Asuncion))
Resto = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Paraguay").filter(ee.Filter.inList("NAME_1", Resto))
San_Pedro = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Paraguay").filter(ee.Filter.inList("NAME_1", San_Pedro))
Caaguazu = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Paraguay").filter(ee.Filter.inList("NAME_1", Caaguazu))
Itapua = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Paraguay").filter(ee.Filter.inList("NAME_1", Itapua))
Alto_Parana = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Paraguay").filter(ee.Filter.inList("NAME_1", Alto_Parana))
Central = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Paraguay").filter(ee.Filter.inList("NAME_1", Central))
Alto_Paraguay = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Paraguay").filter(ee.Filter.inList("NAME_1", Alto_Paraguay))
Boqueron = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Paraguay").filter(ee.Filter.inList("NAME_1", Boqueron)) 

# Samoa (new)
UPOLU = GADM_level1_prep.filterMetadata('NAME_0','equals','Samoa').filter(ee.Filter.inList("NAME_1", UPOLU))
SAVAII = GADM_level1_prep.filterMetadata('NAME_0','equals','Samoa').filter(ee.Filter.inList("NAME_1", SAVAII))

# Sierra Leone
EAST = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Sierra Leone").filter(ee.Filter.inList("NAME_1", EAST))
NORTH = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Sierra Leone").filter(ee.Filter.inList("NAME_1", NORTH))
SOUTH = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Sierra Leone").filter(ee.Filter.inList("NAME_1", SOUTH))
WEST = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Sierra Leone").filter(ee.Filter.inList("NAME_1", WEST))

# Togo
Maritime_full = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Togo").filter(ee.Filter.inList("NAME_1", Maritime))
Plateaux = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Togo").filter(ee.Filter.inList("NAME_1", Plateaux))
Centrale = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Togo").filter(ee.Filter.inList("NAME_1", Centrale))
Kara = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Togo").filter(ee.Filter.inList("NAME_1", Kara))
Savanes = GADM_level1_prep.filterMetadata("NAME_0", "equals", "Togo").filter(ee.Filter.inList("NAME_1", Savanes))

Lomé_Commune_Golfe_Urbain = GADM_level2_prep.filterMetadata("NAME_0", "equals", "Togo").filter(ee.Filter.inList("NAME_2", Lomé_Commune_Golfe_Urbain))

Maritime = ee.FeatureCollection([
    ee.Feature(Maritime_full.geometry().difference(Lomé_Commune_Golfe_Urbain.geometry(), 10))
    .set("NAME_0", "Togo")
    .set("NAME_1", "Maritime")
    .set("HH7", "Maritime")
])


# Tunisia (new)
TUNISIA_DISTRICT_TUNIS = GADM_level1_prep.filterMetadata('NAME_0','equals','Tunisia').filter(ee.Filter.inList("NAME_1", TUNISIA_DISTRICT_TUNIS))
TUNISIA_NORD_EST = GADM_level1_prep.filterMetadata('NAME_0','equals','Tunisia').filter(ee.Filter.inList("NAME_1", TUNISIA_NORD_EST))
TUNISIA_NORD_OUEST = GADM_level1_prep.filterMetadata('NAME_0','equals','Tunisia').filter(ee.Filter.inList("NAME_1", TUNISIA_NORD_OUEST))
TUNISIA_CENTRE_EST = GADM_level1_prep.filterMetadata('NAME_0','equals','Tunisia').filter(ee.Filter.inList("NAME_1", TUNISIA_CENTRE_EST))
TUNISIA_CENTRE_OUEST = GADM_level1_prep.filterMetadata('NAME_0','equals','Tunisia').filter(ee.Filter.inList("NAME_1", TUNISIA_CENTRE_OUEST))
TUNISIA_SUD_EST = GADM_level1_prep.filterMetadata('NAME_0','equals','Tunisia').filter(ee.Filter.inList("NAME_1", TUNISIA_SUD_EST))
TUNISIA_SUD_OUEST = GADM_level1_prep.filterMetadata('NAME_0','equals','Tunisia').filter(ee.Filter.inList("NAME_1", TUNISIA_SUD_OUEST))

# Vietnam (new)
NORTHERN_MIDLANDS_AND_MOUNTAIN = GADM_level1_prep.filterMetadata('NAME_0','equals','Vietnam').filter(ee.Filter.inList("NAME_1", NORTHERN_MIDLANDS_AND_MOUNTAIN))
RED_RIVER_DELTA = GADM_level1_prep.filterMetadata('NAME_0','equals','Vietnam').filter(ee.Filter.inList("NAME_1", RED_RIVER_DELTA))
NORTH_CENTRAL_AND_COASTAL = GADM_level1_prep.filterMetadata('NAME_0','equals','Vietnam').filter(ee.Filter.inList("NAME_1", NORTH_CENTRAL_AND_COASTAL))
CENTRAL_HIGHLANDS = GADM_level1_prep.filterMetadata('NAME_0','equals','Vietnam').filter(ee.Filter.inList("NAME_1", CENTRAL_HIGHLANDS))
SOUTH_EAST = GADM_level1_prep.filterMetadata('NAME_0','equals','Vietnam').filter(ee.Filter.inList("NAME_1", SOUTH_EAST))
MEKONG_RIVER_DELTA = GADM_level1_prep.filterMetadata('NAME_0','equals','Vietnam').filter(ee.Filter.inList("NAME_1", MEKONG_RIVER_DELTA))

# Dominican Republic (new)
OZAMA = GADM_level1_prep.filterMetadata('NAME_0','equals','Dominican Republic').filter(ee.Filter.inList("NAME_1", OZAMA))
VALDESIA = GADM_level1_prep.filterMetadata('NAME_0','equals','Dominican Republic').filter(ee.Filter.inList("NAME_1", VALDESIA))
ENRIQUILLO = GADM_level1_prep.filterMetadata('NAME_0','equals','Dominican Republic').filter(ee.Filter.inList("NAME_1", ENRIQUILLO))
CIBAO_NORDESTE = GADM_level1_prep.filterMetadata('NAME_0','equals','Dominican Republic').filter(ee.Filter.inList("NAME_1", CIBAO_NORDESTE))
CIBAO_NOROESTE = GADM_level1_prep.filterMetadata('NAME_0','equals','Dominican Republic').filter(ee.Filter.inList("NAME_1", CIBAO_NOROESTE))
CIBAO_NORTE = GADM_level1_prep.filterMetadata('NAME_0','equals','Dominican Republic').filter(ee.Filter.inList("NAME_1", CIBAO_NORTE))
CIBAO_SUR = GADM_level1_prep.filterMetadata('NAME_0','equals','Dominican Republic').filter(ee.Filter.inList("NAME_1", CIBAO_SUR))
EL_VALLE = GADM_level1_prep.filterMetadata('NAME_0','equals','Dominican Republic').filter(ee.Filter.inList("NAME_1", EL_VALLE))
HIGUAMO = GADM_level1_prep.filterMetadata('NAME_0','equals','Dominican Republic').filter(ee.Filter.inList("NAME_1", HIGUAMO))
YUMA = GADM_level1_prep.filterMetadata('NAME_0','equals','Dominican Republic').filter(ee.Filter.inList("NAME_1", YUMA))

# Final FCs
Algeria = ee.FeatureCollection([
    ee.Feature(NORD_CENTRE.union().geometry()).set('NAME_0','Algeria').set('NAME_1','NORD CENTRE').set('NAME_1s',NORD_CENTRE.aggregate_array('NAME_1').join('|')).set('GID_1s',NORD_CENTRE.aggregate_array('GID_1').join('|')).set('HH7','NORD_CENTRE'),
    ee.Feature(NORD_EST.union().geometry()).set('NAME_0','Algeria').set('NAME_1','NORD EST').set('NAME_1s',NORD_EST.aggregate_array('NAME_1').join('|')).set('GID_1s',NORD_EST.aggregate_array('GID_1').join('|')).set('HH7','NORD_EST'),
    ee.Feature(NORD_OUEST.union().geometry()).set('NAME_0','Algeria').set('NAME_1','NORD OUEST').set('NAME_1s',NORD_OUEST.aggregate_array('NAME_1').join('|')).set('GID_1s',NORD_OUEST.aggregate_array('GID_1').join('|')).set('HH7','NORD_OUEST'),
    ee.Feature(HAUT_PLATEAU_CENTRE.union().geometry()).set('NAME_0','Algeria').set('NAME_1','HAUT PLATEAU CENTRE').set('NAME_1s',HAUT_PLATEAU_CENTRE.aggregate_array('NAME_1').join('|')).set('GID_1s',HAUT_PLATEAU_CENTRE.aggregate_array('GID_1').join('|')).set('HH7','HAUT_PLATEAU_CENTRE'),
    ee.Feature(HAUT_PLATEAU_EST.union().geometry()).set('NAME_0','Algeria').set('NAME_1','HAUT PLATEAU EST').set('NAME_1s',HAUT_PLATEAU_EST.aggregate_array('NAME_1').join('|')).set('GID_1s',HAUT_PLATEAU_EST.aggregate_array('GID_1').join('|')).set('HH7','HAUT_PLATEAU_EST'),
    ee.Feature(HAUT_PLATEAU_OUEST.union().geometry()).set('NAME_0','Algeria').set('NAME_1','HAUT PLATEAU OUEST').set('NAME_1s',HAUT_PLATEAU_OUEST.aggregate_array('NAME_1').join('|')).set('GID_1s',HAUT_PLATEAU_OUEST.aggregate_array('GID_1').join('|')).set('HH7','HAUT_PLATEAU_OUEST'),
    ee.Feature(SUD.union().geometry()).set('NAME_0','Algeria').set('NAME_1','SUD').set('NAME_1s',SUD.aggregate_array('NAME_1').join('|')).set('GID_1s',SUD.aggregate_array('GID_1').join('|')).set('HH7','SUD'),
    ])
# Azerbaijan final FC
Azerbaijan = ee.FeatureCollection([
    ee.Feature(BAKU.union().geometry()).set('NAME_0','Azerbaijan').set('NAME_1','BAKU').set('HH7','BAKU'),
    ee.Feature(NAKHCHIVAN_AR.union().geometry()).set('NAME_0','Azerbaijan').set('NAME_1','NAKHCHIVAN_AR').set('HH7','NAKHCHIVAN_AR'),
    ee.Feature(ABSHERON_KHIZI.union().geometry()).set('NAME_0','Azerbaijan').set('NAME_1','ABSHERON_KHIZI').set('HH7','ABSHERON_KHIZI'),
    ee.Feature(DAGHLIG_SHIRVAN.union().geometry()).set('NAME_0','Azerbaijan').set('NAME_1','DAGHLIG_SHIRVAN').set('HH7','DAGHLIG_SHIRVAN'),
    ee.Feature(GANJA_DASHKASAN.union().geometry()).set('NAME_0','Azerbaijan').set('NAME_1','GANJA_DASHKASAN').set('HH7','GANJA_DASHKASAN'),
    ee.Feature(KARABAKH.union().geometry()).set('NAME_0','Azerbaijan').set('NAME_1','KARABAKH').set('HH7','KARABAKH'),
    ee.Feature(GAZAKH_TOVUZ.union().geometry()).set('NAME_0','Azerbaijan').set('NAME_1','GAZAKH_TOVUZ').set('HH7','GAZAKH_TOVUZ'),
    ee.Feature(GUBA_KHACHMAZ.union().geometry()).set('NAME_0','Azerbaijan').set('NAME_1','GUBA_KHACHMAZ').set('HH7','GUBA_KHACHMAZ'),
    ee.Feature(LANKARAN_ASTARA.union().geometry()).set('NAME_0','Azerbaijan').set('NAME_1','LANKARAN_ASTARA').set('HH7','LANKARAN_ASTARA'),
    ee.Feature(CENTRAL_ARAN.union().geometry()).set('NAME_0','Azerbaijan').set('NAME_1','CENTRAL_ARAN').set('HH7','CENTRAL_ARAN'),
    ee.Feature(MIL_MUGHAN.union().geometry()).set('NAME_0','Azerbaijan').set('NAME_1','MIL_MUGHAN').set('HH7','MIL_MUGHAN'),
    ee.Feature(SHAKI_ZAGATALA.union().geometry()).set('NAME_0','Azerbaijan').set('NAME_1','SHAKI_ZAGATALA').set('HH7','SHAKI_ZAGATALA'),
    ee.Feature(SHIRVAN_SALYAN.union().geometry()).set('NAME_0','Azerbaijan').set('NAME_1','SHIRVAN_SALYAN').set('HH7','SHIRVAN_SALYAN')
])

Bangladesh = ee.FeatureCollection([
    ee.Feature(Barishal.union().geometry()).set("NAME_0", "Bangladesh").set("NAME_1", "Barishal").set("NAME_1s", Barishal.aggregate_array("NAME_1").join("|")).set("GID_1s", Barishal.aggregate_array("GID_1").join("|")).set("HH7", "Barishal"),
    ee.Feature(Chattogram.union().geometry()).set("NAME_0", "Bangladesh").set("NAME_1", "Chattogram").set("NAME_1s", Chattogram.aggregate_array("NAME_1").join("|")).set("GID_1s", Chattogram.aggregate_array("GID_1").join("|")).set("HH7", "Chattogram"),
    ee.Feature(Dhaka.geometry()).set("NAME_0", "Bangladesh").set("NAME_1", "Dhaka").set("HH7", "Dhaka"),
    ee.Feature(Khulna.union().geometry()).set("NAME_0", "Bangladesh").set("NAME_1", "Khulna").set("NAME_1s", Khulna.aggregate_array("NAME_1").join("|")).set("GID_1s", Khulna.aggregate_array("GID_1").join("|")).set("HH7", "Khulna"),
    ee.Feature(Mymenshing.union().geometry()).set("NAME_0", "Bangladesh").set("NAME_1", "Mymenshing").set("NAME_2s", Mymenshing.aggregate_array("NAME_2").join("|")).set("GID_2s", Mymenshing.aggregate_array("GID_2").join("|")).set("HH7", "Mymenshing"),
    ee.Feature(Rajshahi.union().geometry()).set("NAME_0", "Bangladesh").set("NAME_1", "Rajshahi").set("NAME_1s", Rajshahi.aggregate_array("NAME_1").join("|")).set("GID_1s", Rajshahi.aggregate_array("GID_1").join("|")).set("HH7", "Rajshahi"),
    ee.Feature(Rangpur.union().geometry()).set("NAME_0", "Bangladesh").set("NAME_1", "Rangpur").set("NAME_1s", Rangpur.aggregate_array("NAME_1").join("|")).set("GID_1s", Rangpur.aggregate_array("GID_1").join("|")).set("HH7", "Rangpur"),
    ee.Feature(Sylhet.union().geometry()).set("NAME_0", "Bangladesh").set("NAME_1", "Sylhet").set("NAME_1s", Sylhet.aggregate_array("NAME_1").join("|")).set("GID_1s", Sylhet.aggregate_array("GID_1").join("|")).set("HH7", "Sylhet")
])

CAR = ee.FeatureCollection([
    ee.Feature(REGION_1.union().geometry()).set('NAME_0','Central African Republic').set('NAME_1','REGION_1').set('NAME_1s',REGION_1.aggregate_array('NAME_1').join('|')).set('GID_1s',REGION_1.aggregate_array('GID_1').join('|')).set("HH7","REGION_1"),
    ee.Feature(REGION_2.union().geometry()).set('NAME_0','Central African Republic').set('NAME_1','REGION_2').set('NAME_1s',REGION_2.aggregate_array('NAME_1').join('|')).set('GID_1s',REGION_2.aggregate_array('GID_1').join('|')).set("HH7","REGION_2"),
    ee.Feature(REGION_3.union().geometry()).set('NAME_0','Central African Republic').set('NAME_1','REGION_3').set('NAME_1s',REGION_3.aggregate_array('NAME_1').join('|')).set('GID_1s',REGION_3.aggregate_array('GID_1').join('|')).set("HH7","REGION_3"),
    ee.Feature(REGION_4.union().geometry()).set('NAME_0','Central African Republic').set('NAME_1','REGION_4').set('NAME_1s',REGION_4.aggregate_array('NAME_1').join('|')).set('GID_1s',REGION_4.aggregate_array('GID_1').join('|')).set("HH7","REGION_4"),
    ee.Feature(REGION_5.union().geometry()).set('NAME_0','Central African Republic').set('NAME_1','REGION_5').set('NAME_1s',REGION_5.aggregate_array('NAME_1').join('|')).set('GID_1s',REGION_5.aggregate_array('GID_1').join('|')).set("HH7","REGION_5"),
    ee.Feature(REGION_6.union().geometry()).set('NAME_0','Central African Republic').set('NAME_1','REGION_6').set('NAME_1s',REGION_6.aggregate_array('NAME_1').join('|')).set('GID_1s',REGION_6.aggregate_array('GID_1').join('|')).set("HH7","REGION_6"),
    ee.Feature(REGION_7.union().geometry()).set('NAME_0','Central African Republic').set('NAME_1','REGION_7').set('NAME_1s',REGION_7.aggregate_array('NAME_1').join('|')).set('GID_1s',REGION_7.aggregate_array('GID_1').join('|')).set("HH7","REGION_7")
    ])
Mongolia = ee.FeatureCollection([
    ee.Feature(WESTERN.union().geometry()).set('NAME_0','Mongolia').set('NAME_1','WESTERN').set('NAME_1s',WESTERN.aggregate_array('NAME_1').join('|')).set('GID_1s',WESTERN.aggregate_array('GID_1').join('|')).set('HH7','WESTERN'),
    ee.Feature(EASTERN.union().geometry()).set('NAME_0','Mongolia').set('NAME_1','EASTERN').set('NAME_1s',EASTERN.aggregate_array('NAME_1').join('|')).set('GID_1s',EASTERN.aggregate_array('GID_1').join('|')).set('HH7','EASTERN'),
    ee.Feature(ULAANBAATAR.union().geometry()).set('NAME_0','Mongolia').set('NAME_1','ULAANBAATAR').set('NAME_1s',ULAANBAATAR.aggregate_array('NAME_1').join('|')).set('GID_1s',ULAANBAATAR.aggregate_array('GID_1').join('|')).set('HH7','ULAANBAATAR'),
    ee.Feature(CENTRAL.union().geometry()).set('NAME_0','Mongolia').set('NAME_1','CENTRAL').set('NAME_1s',CENTRAL.aggregate_array('NAME_1').join('|')).set('GID_1s',CENTRAL.aggregate_array('GID_1').join('|')).set('HH7','CENTRAL'),
    ee.Feature(KHANGAI.union().geometry()).set('NAME_0','Mongolia').set('NAME_1','KHANGAI').set('NAME_1s',KHANGAI.aggregate_array('NAME_1').join('|')).set('GID_1s',KHANGAI.aggregate_array('GID_1').join('|')).set('HH7','KHANGAI')
    ])

Paraguay = ee.FeatureCollection([
    ee.Feature(Asuncion.union().geometry()).set("NAME_0", "Paraguay").set("NAME_1", "Asunción").set("NAME_1s", Asuncion.aggregate_array("NAME_1").join("|")).set("GID_1s", Asuncion.aggregate_array("GID_1").join("|")).set("HH7", "Asunción"),
    ee.Feature(Resto.union().geometry()).set("NAME_0", "Paraguay").set("NAME_1", "Resto").set("NAME_1s", Resto.aggregate_array("NAME_1").join("|")).set("GID_1s", Resto.aggregate_array("GID_1").join("|")).set("HH7", "Resto"),
    ee.Feature(San_Pedro.union().geometry()).set("NAME_0", "Paraguay").set("NAME_1", "San Pedro").set("NAME_1s", San_Pedro.aggregate_array("NAME_1").join("|")).set("GID_1s", San_Pedro.aggregate_array("GID_1").join("|")).set("HH7", "San Pedro"),
    ee.Feature(Caaguazu.union().geometry()).set("NAME_0", "Paraguay").set("NAME_1", "Caaguazú").set("NAME_1s", Caaguazu.aggregate_array("NAME_1").join("|")).set("GID_1s", Caaguazu.aggregate_array("GID_1").join("|")).set("HH7", "Caaguazú"),
    ee.Feature(Itapua.union().geometry()).set("NAME_0", "Paraguay").set("NAME_1", "Itapúa").set("NAME_1s", Itapua.aggregate_array("NAME_1").join("|")).set("GID_1s", Itapua.aggregate_array("GID_1").join("|")).set("HH7", "Itapúa"),
    ee.Feature(Alto_Parana.union().geometry()).set("NAME_0", "Paraguay").set("NAME_1", "Alto Paraná").set("NAME_1s", Alto_Parana.aggregate_array("NAME_1").join("|")).set("GID_1s", Alto_Parana.aggregate_array("GID_1").join("|")).set("HH7", "Alto Paraná"),
    ee.Feature(Central.union().geometry()).set("NAME_0", "Paraguay").set("NAME_1", "Central").set("NAME_1s", Central.aggregate_array("NAME_1").join("|")).set("GID_1s", Central.aggregate_array("GID_1").join("|")).set("HH7", "Central"),
    ee.Feature(Alto_Paraguay.union().geometry()).set("NAME_0", "Paraguay").set("NAME_1", "Alto Paraguay").set("NAME_1s", Alto_Paraguay.aggregate_array("NAME_1").join("|")).set("GID_1s", Alto_Paraguay.aggregate_array("GID_1").join("|")).set("HH7", "Alto Paraguay"),
    ee.Feature(Boqueron.union().geometry()).set("NAME_0", "Paraguay").set("NAME_1", "Boquerón").set("NAME_1s", Boqueron.aggregate_array("NAME_1").join("|")).set("GID_1s", Boqueron.aggregate_array("GID_1").join("|")).set("HH7", "Boquerón")
])

Sierra_Leone = ee.FeatureCollection([
    ee.Feature(EAST.union().geometry()).set("NAME_0", "Sierra Leone").set("NAME_1", "EAST").set("NAME_1s", EAST.aggregate_array("NAME_1").join("|")).set("GID_1s", EAST.aggregate_array("GID_1").join("|")).set("HH7", "EAST"),
    ee.Feature(NORTH.union().geometry()).set("NAME_0", "Sierra Leone").set("NAME_1", "NORTH").set("NAME_1s", NORTH.aggregate_array("NAME_1").join("|")).set("GID_1s", NORTH.aggregate_array("GID_1").join("|")).set("HH7", "NORTH"),
    ee.Feature(SOUTH.union().geometry()).set("NAME_0", "Sierra Leone").set("NAME_1", "SOUTH").set("NAME_1s", SOUTH.aggregate_array("NAME_1").join("|")).set("GID_1s", SOUTH.aggregate_array("GID_1").join("|")).set("HH7", "SOUTH"),
    ee.Feature(WEST.union().geometry()).set("NAME_0", "Sierra Leone").set("NAME_1", "WEST").set("NAME_1s", WEST.aggregate_array("NAME_1").join("|")).set("GID_1s", WEST.aggregate_array("GID_1").join("|")).set("HH7", "WEST")
])

Togo = ee.FeatureCollection([
    ee.Feature(Maritime.geometry()).set("NAME_0", "Togo").set("NAME_1", "Maritime").set("HH7", "Maritime"),
    ee.Feature(Plateaux.union().geometry()).set("NAME_0", "Togo").set("NAME_1", "Plateaux").set("NAME_1s", Plateaux.aggregate_array("NAME_1").join("|")).set("GID_1s", Plateaux.aggregate_array("GID_1").join("|")).set("HH7", "Plateaux"),
    ee.Feature(Centrale.union().geometry()).set("NAME_0", "Togo").set("NAME_1", "Centrale").set("NAME_1s", Centrale.aggregate_array("NAME_1").join("|")).set("GID_1s", Centrale.aggregate_array("GID_1").join("|")).set("HH7", "Centrale"),
    ee.Feature(Kara.union().geometry()).set("NAME_0", "Togo").set("NAME_1", "Kara").set("NAME_1s", Kara.aggregate_array("NAME_1").join("|")).set("GID_1s", Kara.aggregate_array("GID_1").join("|")).set("HH7", "Kara"),
    ee.Feature(Savanes.union().geometry()).set("NAME_0", "Togo").set("NAME_1", "Savanes").set("NAME_1s", Savanes.aggregate_array("NAME_1").join("|")).set("GID_1s", Savanes.aggregate_array("GID_1").join("|")).set("HH7", "Savanes"),
    ee.Feature(Lomé_Commune_Golfe_Urbain.union().geometry()).set("NAME_0", "Togo").set("NAME_1", "Lomé Commune/Golfe Urbain").set("NAME_2s", Lomé_Commune_Golfe_Urbain.aggregate_array("NAME_2").join("|")).set("GID_2s", Lomé_Commune_Golfe_Urbain.aggregate_array("GID_2").join("|")).set("HH7", "Lomé Commune/Golfe Urbain")
])

Tunisia = ee.FeatureCollection([
    ee.Feature(TUNISIA_DISTRICT_TUNIS.union().geometry()).set('NAME_0','Tunisia').set('NAME_1','TUNISIA_DISTRICT_TUNIS').set('NAME_1s', TUNISIA_DISTRICT_TUNIS.aggregate_array('NAME_1').join('|')).set('GID_1s', TUNISIA_DISTRICT_TUNIS.aggregate_array('GID_1').join('|')).set('HH7','TUNISIA_DISTRICT_TUNIS'),
    ee.Feature(TUNISIA_NORD_EST.union().geometry()).set('NAME_0','Tunisia').set('NAME_1','TUNISIA_NORD_EST').set('NAME_1s', TUNISIA_NORD_EST.aggregate_array('NAME_1').join('|')).set('GID_1s', TUNISIA_NORD_EST.aggregate_array('GID_1').join('|')).set('HH7','TUNISIA_NORD_EST'),
    ee.Feature(TUNISIA_NORD_OUEST.union().geometry()).set('NAME_0','Tunisia').set('NAME_1','TUNISIA_NORD_OUEST').set('NAME_1s', TUNISIA_NORD_OUEST.aggregate_array('NAME_1').join('|')).set('GID_1s', TUNISIA_NORD_OUEST.aggregate_array('GID_1').join('|')).set('HH7','TUNISIA_NORD_OUEST'),
    ee.Feature(TUNISIA_CENTRE_EST.union().geometry()).set('NAME_0','Tunisia').set('NAME_1','TUNISIA_CENTRE_EST').set('NAME_1s', TUNISIA_CENTRE_EST.aggregate_array('NAME_1').join('|')).set('GID_1s', TUNISIA_CENTRE_EST.aggregate_array('GID_1').join('|')).set('HH7','TUNISIA_CENTRE_EST'),
    ee.Feature(TUNISIA_CENTRE_OUEST.union().geometry()).set('NAME_0','Tunisia').set('NAME_1','TUNISIA_CENTRE_OUEST').set('NAME_1s', TUNISIA_CENTRE_OUEST.aggregate_array('NAME_1').join('|')).set('GID_1s', TUNISIA_CENTRE_OUEST.aggregate_array('GID_1').join('|')).set('HH7','TUNISIA_CENTRE_OUEST'),
    ee.Feature(TUNISIA_SUD_EST.union().geometry()).set('NAME_0','Tunisia').set('NAME_1','TUNISIA_SUD_EST').set('NAME_1s', TUNISIA_SUD_EST.aggregate_array('NAME_1').join('|')).set('GID_1s', TUNISIA_SUD_EST.aggregate_array('GID_1').join('|')).set('HH7','TUNISIA_SUD_EST'),
    ee.Feature(TUNISIA_SUD_OUEST.union().geometry()).set('NAME_0','Tunisia').set('NAME_1','TUNISIA_SUD_OUEST').set('NAME_1s', TUNISIA_SUD_OUEST.aggregate_array('NAME_1').join('|')).set('GID_1s', TUNISIA_SUD_OUEST.aggregate_array('GID_1').join('|')).set('HH7','TUNISIA_SUD_OUEST')
])
Samoa = ee.FeatureCollection([
    ee.Feature(UPOLU.union().geometry()).set('NAME_0','Samoa').set('NAME_1','UPOLU').set('NAME_1s', UPOLU.aggregate_array('NAME_1').join('|')).set('GID_1s', UPOLU.aggregate_array('GID_1').join('|')).set('HH7','UPOLU'),
    ee.Feature(SAVAII.union().geometry()).set('NAME_0','Samoa').set('NAME_1','SAVAII').set('NAME_1s', SAVAII.aggregate_array('NAME_1').join('|')).set('GID_1s', SAVAII.aggregate_array('GID_1').join('|')).set('HH7','SAVAII')
])

Vietnam = ee.FeatureCollection([
    ee.Feature(NORTHERN_MIDLANDS_AND_MOUNTAIN.union().geometry()).set('NAME_0','Vietnam').set('NAME_1','NORTHERN_MIDLANDS_AND_MOUNTAIN').set('NAME_1s', NORTHERN_MIDLANDS_AND_MOUNTAIN.aggregate_array('NAME_1').join('|')).set('GID_1s', NORTHERN_MIDLANDS_AND_MOUNTAIN.aggregate_array('GID_1').join('|')).set('HH7','NORTHERN_MIDLANDS_AND_MOUNTAIN'),
    ee.Feature(RED_RIVER_DELTA.union().geometry()).set('NAME_0','Vietnam').set('NAME_1','RED_RIVER_DELTA').set('NAME_1s', RED_RIVER_DELTA.aggregate_array('NAME_1').join('|')).set('GID_1s', RED_RIVER_DELTA.aggregate_array('GID_1').join('|')).set('HH7','RED_RIVER_DELTA'),
    ee.Feature(NORTH_CENTRAL_AND_COASTAL.union().geometry()).set('NAME_0','Vietnam').set('NAME_1','NORTH_CENTRAL_AND_COASTAL').set('NAME_1s', NORTH_CENTRAL_AND_COASTAL.aggregate_array('NAME_1').join('|')).set('GID_1s', NORTH_CENTRAL_AND_COASTAL.aggregate_array('GID_1').join('|')).set('HH7','NORTH_CENTRAL_AND_COASTAL'),
    ee.Feature(CENTRAL_HIGHLANDS.union().geometry()).set('NAME_0','Vietnam').set('NAME_1','CENTRAL_HIGHLANDS').set('NAME_1s', CENTRAL_HIGHLANDS.aggregate_array('NAME_1').join('|')).set('GID_1s', CENTRAL_HIGHLANDS.aggregate_array('GID_1').join('|')).set('HH7','CENTRAL_HIGHLANDS'),
    ee.Feature(SOUTH_EAST.union().geometry()).set('NAME_0','Vietnam').set('NAME_1','SOUTH_EAST').set('NAME_1s', SOUTH_EAST.aggregate_array('NAME_1').join('|')).set('GID_1s', SOUTH_EAST.aggregate_array('GID_1').join('|')).set('HH7','SOUTH_EAST'),
    ee.Feature(MEKONG_RIVER_DELTA.union().geometry()).set('NAME_0','Vietnam').set('NAME_1','MEKONG_RIVER_DELTA').set('NAME_1s', MEKONG_RIVER_DELTA.aggregate_array('NAME_1').join('|')).set('GID_1s', MEKONG_RIVER_DELTA.aggregate_array('GID_1').join('|')).set('HH7','MEKONG_RIVER_DELTA')
])

Dominican_Republic = ee.FeatureCollection([
    ee.Feature(OZAMA.union().geometry()).set('NAME_0','Dominican Republic').set('NAME_1','OZAMA').set('NAME_1s', OZAMA.aggregate_array('NAME_1').join('|')).set('GID_1s', OZAMA.aggregate_array('GID_1').join('|')).set('HH7','OZAMA'),
    ee.Feature(VALDESIA.union().geometry()).set('NAME_0','Dominican Republic').set('NAME_1','VALDESIA').set('NAME_1s', VALDESIA.aggregate_array('NAME_1').join('|')).set('GID_1s', VALDESIA.aggregate_array('GID_1').join('|')).set('HH7','VALDESIA'),
    ee.Feature(ENRIQUILLO.union().geometry()).set('NAME_0','Dominican Republic').set('NAME_1','ENRIQUILLO').set('NAME_1s', ENRIQUILLO.aggregate_array('NAME_1').join('|')).set('GID_1s', ENRIQUILLO.aggregate_array('GID_1').join('|')).set('HH7','ENRIQUILLO'),
    ee.Feature(CIBAO_NORDESTE.union().geometry()).set('NAME_0','Dominican Republic').set('NAME_1','CIBAO_NORDESTE').set('NAME_1s', CIBAO_NORDESTE.aggregate_array('NAME_1').join('|')).set('GID_1s', CIBAO_NORDESTE.aggregate_array('GID_1').join('|')).set('HH7','CIBAO_NORDESTE'),
    ee.Feature(CIBAO_NOROESTE.union().geometry()).set('NAME_0','Dominican Republic').set('NAME_1','CIBAO_NOROESTE').set('NAME_1s', CIBAO_NOROESTE.aggregate_array('NAME_1').join('|')).set('GID_1s', CIBAO_NOROESTE.aggregate_array('GID_1').join('|')).set('HH7','CIBAO_NOROESTE'),
    ee.Feature(CIBAO_NORTE.union().geometry()).set('NAME_0','Dominican Republic').set('NAME_1','CIBAO_NORTE').set('NAME_1s', CIBAO_NORTE.aggregate_array('NAME_1').join('|')).set('GID_1s', CIBAO_NORTE.aggregate_array('GID_1').join('|')).set('HH7','CIBAO_NORTE'),
    ee.Feature(CIBAO_SUR.union().geometry()).set('NAME_0','Dominican Republic').set('NAME_1','CIBAO_SUR').set('NAME_1s', CIBAO_SUR.aggregate_array('NAME_1').join('|')).set('GID_1s', CIBAO_SUR.aggregate_array('GID_1').join('|')).set('HH7','CIBAO_SUR'),
    ee.Feature(EL_VALLE.union().geometry()).set('NAME_0','Dominican Republic').set('NAME_1','EL_VALLE').set('NAME_1s', EL_VALLE.aggregate_array('NAME_1').join('|')).set('GID_1s', EL_VALLE.aggregate_array('GID_1').join('|')).set('HH7','EL_VALLE'),
    ee.Feature(HIGUAMO.union().geometry()).set('NAME_0','Dominican Republic').set('NAME_1','HIGUAMO').set('NAME_1s', HIGUAMO.aggregate_array('NAME_1').join('|')).set('GID_1s', HIGUAMO.aggregate_array('GID_1').join('|')).set('HH7','HIGUAMO'),
    ee.Feature(YUMA.union().geometry()).set('NAME_0','Dominican Republic').set('NAME_1','YUMA').set('NAME_1s', YUMA.aggregate_array('NAME_1').join('|')).set('GID_1s', YUMA.aggregate_array('GID_1').join('|')).set('HH7','YUMA')
])

GADM_level1 = (
    GADM_level1
    .merge(Algeria)
    .merge(Bangladesh)
    .merge(CAR)
    .merge(Sierra_Leone)
    .merge(Tunisia)
    .merge(Togo)
    .merge(Mongolia)
    .merge(Paraguay)
    .merge(Vietnam)
    .merge(Dominican_Republic)
    .merge(Azerbaijan)
    .merge(Samoa)
)


# Add the missing properties 
GADM_level3 = GADM_level3.map(lambda f: f.set('NAME_1s', 'NaN').set('GID_1s', 'NaN'))
GADM_level0 = GADM_level0.map(lambda f: f.set(ee.Dictionary.fromLists(                                                                
      p := GADM_level3.first().propertyNames().removeAll(GADM_level0.first().propertyNames()),                                          
      ee.List.repeat('NaN', p.size())                                                                                                   
  )))             
GADM_level2 = GADM_level2.map(lambda f: f.set(ee.Dictionary.fromLists(                                                                
      p := GADM_level3.first().propertyNames().removeAll(GADM_level2.first().propertyNames()),
      ee.List.repeat('NaN', p.size())                                                                                                   
  )))                                                                                                                                   
GADM_level1 = GADM_level1.map(lambda f: f.set(ee.Dictionary.fromLists(                                                                
      p := GADM_level3.first().propertyNames().removeAll(GADM_level1.first().propertyNames()),
      ee.List.repeat('NaN', p.size())                                                                                                   
  )))

# Merge the data 
GADM_toSample = GADM_level2.merge(GADM_level0).merge(GADM_level3).merge(GADM_level1)
# print(GADM_toSample)

##/ Define the new regions for Level 3
Pakistan_Gujranwala = ee.FeatureCollection(                        
      ee.Feature(GADM_level3_prep.filterMetadata('NAME_3', 'contains', 'Gujranwala').geometry())
      .set('NAME_0', 'Pakistan')                                                                                                        
      .set('NAME_3', 'Gujranwala')                                                                                                      
      .set('HH7', 'Gujranwala')                                                                                                         
  ).map(lambda f: f.set(ee.Dictionary.fromLists(                                                                                        
      p := GADM_level3_prep.first().propertyNames().removeAll(['NAME_0', 'NAME_3']),                                                    
      ee.List.repeat('NaN', p.size())                                                                                                   
  )))             
                                                                                                                                        
REGIAO_NORTE_OESTE = ee.FeatureCollection(
      ee.Feature(GADM_level2_prep.filterMetadata('NAME_0', 'contains', 'São Tomé and Príncipe')
      .filter(ee.Filter.inList('NAME_2', ['Lembá', 'Lobata'])).geometry())                                                              
      .set('NAME_0', 'São Tomé and Príncipe')
      .set('NAME_2', 'REGIÃO NORTE OESTE')                                                                                              
      .set('HH7', 'REGIAO_NORTE_OESTE')
  ).map(lambda f: f.set(ee.Dictionary.fromLists(                                                                                        
      p := GADM_level2_prep.first().propertyNames().removeAll(['NAME_0', 'NAME_2']),                                                    
      ee.List.repeat('NaN', p.size())
  )))                                                                                                                                   
                  
REGIAO_SUL_ESTE = ee.FeatureCollection(                                                                                               
      ee.Feature(GADM_level2_prep.filterMetadata('NAME_0', 'contains', 'São Tomé and Príncipe')
      .filter(ee.Filter.inList('NAME_2', ['Cantagalo', 'Caué'])).geometry())                                                            
      .set('NAME_0', 'São Tomé and Príncipe')                                                                                           
      .set('NAME_2', 'REGIÃO SUL ESTE')
      .set('HH7', 'REGIAO_SUL_ESTE')                                                                                                    
  ).map(lambda f: f.set(ee.Dictionary.fromLists(                                                                                        
      p := GADM_level2_prep.first().propertyNames().removeAll(['NAME_0', 'NAME_2']),
      ee.List.repeat('NaN', p.size())                                                                                                   
  )))             
                                                                                                                                        
Principe = ee.FeatureCollection(
      ee.Feature(GADM_level1_prep.filterMetadata('NAME_0', 'contains', 'São Tomé and Príncipe')
      .filter(ee.Filter.inList('NAME_1', ['Príncipe'])).geometry())                                                                     
      .set('NAME_0', 'São Tomé and Príncipe')
      .set('NAME_1', 'Príncipe')                                                                                                        
      .set('HH7', 'Príncipe')                                                                                                           
  ).map(lambda f: f.set(ee.Dictionary.fromLists(
      p := GADM_level1_prep.first().propertyNames().removeAll(['NAME_0', 'NAME_1']),                                                    
      ee.List.repeat('NaN', p.size())                                                                                                   
  )))

# Add them to the to sampling data 
GADM_toSample = GADM_toSample.merge(Pakistan_Gujranwala).merge(REGIAO_NORTE_OESTE).merge(REGIAO_SUL_ESTE).merge(Principe)

# filter countries from GADM that are in MICS or DHS datasets
#countries_in_DHS = dhs_gdf['CNTRYNAMEE'].unique().tolist()
#countries_in_MICS = mics_gdf['CNTRYNAME'].unique().tolist()
#countries_to_exclude = set(countries_in_DHS + countries_in_MICS)
#GADM_toSample = GADM_toSample.filter(ee.Filter.inList('NAME_0', list(countries_to_exclude)).Not())

# Exclude countries that already have MICS boundary files

mics_country_name_map = {
    "Lao People's Democratic Republic": "Laos",
    "Eswatini": "Swaziland",  # only if GADM uses Swaziland in your file
}

mics_countries_raw = set(mics_gdf["CNTRYNAME"].dropna().unique())

mics_countries_for_gadm = {
    mics_country_name_map.get(country, country)
    for country in mics_countries_raw
}

mics_countries_for_gadm = set(mics_countries_for_gadm)

# Filter countries with MICS boundaries

print("\nMICS countries to exclude from GADM_toSample:")
for country in sorted(mics_countries_for_gadm):
    print(country)

GADM_toSample = GADM_toSample.filter(
    ee.Filter.inList("NAME_0", list(mics_countries_for_gadm)).Not()
)


MICS countries to exclude from GADM_toSample:
Gambia
Georgia
Ghana
Honduras
Kosovo
Laos
Lesotho
Madagascar
Malawi
Nigeria
Sierra Leone
Swaziland
Zimbabwe


In [7]:
# Below list of countries that need manual assignment of survey year 
analysisYearByCountry = {
    'Algeria': 2018,
    'Bangladesh': 2019,
    'Central African Republic': 2018,
    'Chad': 2019,
    #'Fiji': 2021,
    'Gambia': 2018,
    'Georgia': 2018,
    'Ghana': 2017,
    'Guinea-Bissau': 2018,
    'Guyana': 2019,
    'Iraq': 2018,
    'Kiribati': 2018,
    'Kosovo': 2019,
    'Laos': 2023,
    'Lesotho': 2018,
    'Madagascar': 2018,
    'Mongolia': 2018,
    'Nigeria': 2016,
    'Pakistan': 2017,
    'Palestina': 2019,
    'Paraguay': 2016,
    'Sierra Leone': 2017,
    'Suriname': 2018,
    'São Tomé and Príncipe': 2019,
    'Togo': 2017,
    'Tonga': 2019,
    'Tunisia': 2023,
    'Zimbabwe': 2019,

    'Azerbaijan': 2023,
    'Democratic Republic of the Congo': 2017, 
    'Benin': 2021,
    'Samoa': 2019,
    'Tuvalu': 2019,
    'Vanuatu': 2023,
    'Vietnam': 2020,
    'Dominican Republic': 2019

}

# Get year from dictionary and assign to each feature
year_dict = ee.Dictionary(analysisYearByCountry)                                                                                      
                                                                                                                                    
def assign_year(feature):                                                                                                             
    year = ee.Number(year_dict.get(feature.getString('NAME_0'), ee.Number(2018)))                                                     
    return feature.set('analysis_year', year)                                                                                         
                
GADM_toSample = GADM_toSample.map(assign_year)   

# print countries not in the dictionary
countries_in_GADM = GADM_toSample.aggregate_array('NAME_0').distinct().getInfo()
countries_not_in_dict = set(countries_in_GADM).difference(set(analysisYearByCountry.keys()))
# If empty, print all good, else print the list of countries
if not countries_not_in_dict:
    print("All countries in GADM_toSample have an assigned analysis year.")
else:
    print("The following countries in GADM_toSample do NOT have an assigned analysis year:")
    for country in countries_not_in_dict:
        print(country)

# Summary of countries and number of regions in GADM_toSample
gadm_summary = (
    GADM_toSample
    .aggregate_histogram("NAME_0")
    .getInfo()
)

print("Countries and number of GADM-derived survey regions to export:")
for country, n_regions in sorted(gadm_summary.items()):
    print(f"{country}: {n_regions}")

print(f"\nTotal countries: {len(gadm_summary)}")
print(f"Total regions: {sum(gadm_summary.values())}")

#Export files
out_dir = "outputs/02_boundary_files/gadm_by_country"
os.makedirs(out_dir, exist_ok=True)

countries = sorted(GADM_toSample.aggregate_array("NAME_0").distinct().getInfo())

for country in countries:
    safe_country = (
        country.replace(" ", "_")
        .replace("/", "_")
        .replace("ã", "a")
        .replace("é", "e")
        .replace("í", "i")
        .replace("ó", "o")
        .replace("ç", "c")
    )

    out_path = f"{out_dir}/{safe_country}.shp"

    if os.path.exists(out_path):
        print(f"Skipping {country}, already exists.")
        continue

    # remove failed/corrupt partial files from previous attempts
    for ext in [".shp", ".shx", ".dbf", ".prj", ".cpg", ".zip"]:
        p = Path(out_path).with_suffix(ext)
        if p.exists():
            p.unlink()

    print(f"Exporting {country}...")
    fc_country = GADM_toSample.filter(ee.Filter.eq("NAME_0", country))

    try:
        geemap.ee_export_vector(
            fc_country,
            filename=out_path,
            timeout=1000
        )
    except Exception as e:
        print(f"FAILED: {country}")
        print(e)

        

All countries in GADM_toSample have an assigned analysis year.
Countries and number of GADM-derived survey regions to export:
Algeria: 7
Azerbaijan: 13
Bangladesh: 8
Benin: 12
Central African Republic: 7
Chad: 23
Democratic Republic of the Congo: 26
Dominican Republic: 10
Guinea-Bissau: 9
Guyana: 10
Iraq: 18
Kiribati: 5
Mongolia: 5
Pakistan: 142
Palestina: 16
Paraguay: 9
Samoa: 2
Suriname: 10
São Tomé and Príncipe: 5
Togo: 6
Tonga: 5
Tunisia: 7
Tuvalu: 9
Vanuatu: 6
Vietnam: 6

Total countries: 25
Total regions: 376
Skipping Algeria, already exists.
Skipping Azerbaijan, already exists.
Skipping Bangladesh, already exists.
Skipping Benin, already exists.
Skipping Central African Republic, already exists.
Skipping Chad, already exists.
Skipping Democratic Republic of the Congo, already exists.
Skipping Dominican Republic, already exists.
Skipping Guinea-Bissau, already exists.
Skipping Guyana, already exists.
Skipping Iraq, already exists.
Skipping Kiribati, already exists.
Skipping Mongo

### 1b. Other Surveys Configuration

Defines boundary construction and defines analysis-year for the other (non-water-quality) MICS surveys.

In [52]:
# ============================================================
# OTHER / MICS WITHOUT WATER QUALITY — boundary construction
# Uses final_match_status_otherMICS_1.csv comments as guidance
# ============================================================

# Requires:
# GADM_level1_prep
# GADM_level2_prep
# ee

countryLevel1_other = ['Afghanistan','Costa Rica']

GADM_level1_other = GADM_level1_prep.filter(ee.Filter.inList('NAME_0',countryLevel1_other)) \
                                  .map(lambda f: f.set('HH7',f.getString('NAME_1')))
# -------------------------
# Helpers
# -------------------------

def fc_l1(country, names):
    return (
        GADM_level1_prep
        .filterMetadata("NAME_0", "equals", country)
        .filter(ee.Filter.inList("NAME_1", names))
    )

def fc_l2(country, names):
    return (
        GADM_level2_prep
        .filterMetadata("NAME_0", "equals", country)
        .filter(ee.Filter.inList("NAME_2", names))
    )

def make_feature(fc, country, name, hh7, level=1):
    return (
        ee.Feature(fc.geometry())
        .set("NAME_0", country)
        .set("NAME_1", name)
        .set("HH7", hh7)
        .set(f"NAME_{level}s", fc.aggregate_array(f"NAME_{level}").join("|"))
        .set(f"GID_{level}s", fc.aggregate_array(f"GID_{level}").join("|"))
    )

def make_country_l1(country, region_map):
    return ee.FeatureCollection([
        make_feature(fc_l1(country, gadm_names), country, hh7, hh7, level=1)
        for hh7, gadm_names in region_map.items()
    ])

def make_country_l2(country, region_map):
    return ee.FeatureCollection([
        make_feature(fc_l2(country, gadm_names), country, hh7, hh7, level=2)
        for hh7, gadm_names in region_map.items()
    ])

def make_direct_country(country, mapping, level=1):
    if level == 1:
        return make_country_l1(country, mapping)
    if level == 2:
        return make_country_l2(country, mapping)


# ============================================================
# Comoros
# ============================================================

Comoros = make_country_l1("Comoros", {
    "Mwali_Comoros": ["Mwali"],
    "Ndzuwani_Comoros": ["Nzwani"],
    "Ngazidja_Comoros": ["Njazídja"],
})
GADM_level1_other = GADM_level1_other.merge(Comoros)



# ============================================================
# Nauru — ignore LOCATION because CSV says likely MICS error
# ============================================================

Nauru = make_country_l1("Nauru", {
    "YAREN_Nauru": ["Yaren"],
    "BOE_Nauru": ["Boe"],
    "AIWO_Nauru": ["Aiwo"],
    "BUADA_Nauru": ["Buada"],
    "DENIGOMODU_Nauru": ["Denigomodu"],
    "NIBOK_Nauru": ["Nibok"],
    "UABOE_Nauru": ["Uaboe"],
    "BAITSI_Nauru": ["Baiti"],
    "EWA_Nauru": ["Ewa"],
    "ANETAN_Nauru": ["Anetan"],
    "ANABAR_Nauru": ["Anabar"],
    "IJUW_Nauru": ["Ijuw"],
    "ANIBARE_Nauru": ["Anibare"],
    "MENENG_Nauru": ["Meneng"],
})
GADM_level1_other = GADM_level1_other.merge(Nauru)




# ============================================================
# Turkmenistan — merge Arkadag + Ahal into one Ahal region
# ============================================================

Turkmenistan = make_country_l1("Turkmenistan", {
    "ASHGABAT_CITY_Turkmenistan": ["Aşgabat"],
    "AKHAL_ARKADAG_Turkmenistan": ["Ahal"],
    "BALKAN_Turkmenistan": ["Balkan"],
    "DASHOGUZ_Turkmenistan": ["Tashauz"],
    "LEBAP_Turkmenistan": ["Chardzhou"],
    "MARY_Turkmenistan": ["Mary"],
})
GADM_level1_other = GADM_level1_other.merge(Turkmenistan)


# ============================================================
# Argentina — grouped regions from CSV comments
# ============================================================

Argentina = make_country_l1("Argentina", {
    "CIUDAD_Y_PARTIDOS_DE_BUENOS_AIRES_Argentina": ["Ciudad de Buenos Aires"],
    "RESTO_PROVINCIA_DE_BUENOS_AIRES_Argentina": ["Buenos Aires"],
    "CUYO_Argentina": ["San Juan", "San Luis", "Mendoza"],
    "NOA_Argentina": ["Jujuy", "Salta", "Santiago del Estero", "Tucumán", "Catamarca", "La Rioja"],
    "NEA_Argentina": ["Formosa", "Chaco", "Corrientes", "Misiones"],
    "PAMPEANA_Argentina": ["La Pampa", "Córdoba", "Santa Fe", "Entre Ríos"],
    "PATAGONICA_Argentina": ["Neuquén", "Río Negro", "Chubut", "Santa Cruz", "Tierra del Fuego"],
})
GADM_level1_other = GADM_level1_other.merge(Argentina)


# ============================================================
# Belarus — Minsk city separated from Minsk region
# ============================================================

Belarus_standard = ee.FeatureCollection([
    make_feature(fc_l1("Belarus", ["Brest"]), "Belarus", "BREST_Belarus", "BREST_Belarus", level=1),
    make_feature(fc_l1("Belarus", ["Vitsyebsk"]), "Belarus", "VITEBSK_Belarus", "VITEBSK_Belarus", level=1),
    make_feature(fc_l1("Belarus", ["Homyel'"]), "Belarus", "GOMEL_Belarus", "GOMEL_Belarus", level=1),
    make_feature(fc_l1("Belarus", ["Hrodna"]), "Belarus", "GRODNO_Belarus", "GRODNO_Belarus", level=1),
    make_feature(fc_l1("Belarus", ["Mahilyow"]), "Belarus", "MOGILEV_Belarus", "MOGILEV_Belarus", level=1),
])

MINSK_L1 = fc_l1("Belarus", ["Minsk"])
MINSK_CITY_L2 = (
    GADM_level2_prep
    .filterMetadata("NAME_0", "equals", "Belarus")
    .filterMetadata("NAME_1", "equals", "Minsk")
    .filterMetadata("NAME_2", "equals", "Minsk")
)

MINSK_CITY_Belarus = ee.FeatureCollection([
    make_feature(MINSK_CITY_L2, "Belarus", "MINSK_CITY_Belarus", "MINSK_CITY_Belarus", level=2)
])

MINSK_Belarus = ee.FeatureCollection([
    ee.Feature(MINSK_L1.geometry().difference(MINSK_CITY_L2.geometry(), 10))
    .set("NAME_0", "Belarus")
    .set("NAME_1", "MINSK_Belarus")
    .set("HH7", "MINSK_Belarus")
    .set("NAME_1s", MINSK_L1.aggregate_array("NAME_1").join("|"))
    .set("GID_1s", MINSK_L1.aggregate_array("GID_1").join("|"))
])

Belarus = Belarus_standard.merge(MINSK_CITY_Belarus).merge(MINSK_Belarus)
GADM_level1_other = GADM_level1_other.merge(Belarus)


# ============================================================
# Cuba — updated Artemisa and Pinar del rio construction
# ============================================================

Cuba_standard_l1 = make_country_l1("Cuba", {
    "LA_HABANA_Cuba": ["Ciudad de la Habana"],
    "MAYABEQUE_Cuba": ["Mayabeque"],
    "VILLA_CLARA_Cuba": ["Villa Clara"],
    "CIENFUEGOS_Cuba": ["Cienfuegos"],
    "GRANMA_Cuba": ["Granma"],
    "MATANZAS_Cuba": ["Matanzas"],
    "SANCTI_SPIRITUS_Cuba": ["Sancti Spíritus"],
    "CIEGO_DE_AVILA_Cuba": ["Ciego de Ávila"],
    "CAMAGUEY_Cuba": ["Camagüey"],
    "LAS_TUNAS_Cuba": ["Las Tunas"],
    "HOLGUIN_Cuba": ["Holguín"],
    "SANTIAGO_DE_CUBA_Cuba": ["Santiago de Cuba"],
    "GUANTANAMO_Cuba": ["Guantánamo"],
    "ISLA_DE_LA_JUVENTUD_Cuba": ["Isla de la Juventud"],
})

PINAR_DEL_RIO_names = [
    "La Palma",
    "Viñales",
    "Minas de Matahambre",
    "Los Palacios",
    "Consolación del Sur",
    "Pinar del Río",
    "Mantua",
    "San Juan y Martínez",
    "San Luis",
    "Guane",
    "Sandino"
]

PINAR_DEL_RIO_fc = (
    GADM_level2_prep
    .filterMetadata("NAME_0", "equals", "Cuba")
    .filterMetadata("NAME_1", "equals", "Pinar del Río")
    .filter(ee.Filter.inList("NAME_2", PINAR_DEL_RIO_names))
)

PINAR_DEL_RIO_Cuba = ee.FeatureCollection([
    make_feature(
        PINAR_DEL_RIO_fc,
        "Cuba",
        "PINAR_DEL_RIO_Cuba",
        "PINAR_DEL_RIO_Cuba",
        level=2
    )
])

ARTEMISA_Cuba = make_country_l2("Cuba", {
    "ARTEMISA_Cuba": [
        "Artemisa", "Mariel", "Guanajay", "Caimito", "Bauta",
        "San Antonio de los Baños", "Güira de Melena", "Alquízar",
        "Bahía Honda", "Candelaria", "San Cristóbal"
    ]
})

Cuba = Cuba_standard_l1.merge(ARTEMISA_Cuba).merge(PINAR_DEL_RIO_Cuba)
GADM_level1_other = GADM_level1_other.merge(Cuba)


# ============================================================
# Jamaica
# ============================================================

Jamaica = make_country_l1("Jamaica", {
    "KINGSTON_Jamaica": ["Kingston"],
    "ST_ANDREW_Jamaica": ["Saint Andrew"],
    "ST_CATHERINE_Jamaica": ["Saint Catherine"],
    "ST_THOMAS_Jamaica": ["Saint Thomas"],
    "PORTLAND_Jamaica": ["Portland"],
    "ST_MARY_Jamaica": ["Saint Mary"],
    "ST_ANN_Jamaica": ["Saint Ann"],
    "TRELAWNY_Jamaica": ["Trelawny"],
    "ST_JAMES_Jamaica": ["Saint James"],
    "HANOVER_Jamaica": ["Hanover"],
    "WESTMORELAND_Jamaica": ["Westmoreland"],
    "ST_ELIZABETH_Jamaica": ["Saint Elizabeth"],
    "MANCHESTER_Jamaica": ["Manchester"],
    "CLARENDON_Jamaica": ["Clarendon"],
})
GADM_level1_other = GADM_level1_other.merge(Jamaica)


# ============================================================
# Montenegro
# ============================================================

Montenegro = make_country_l1("Montenegro", {
    "SOUTH_Montenegro": ["Herceg Novi", "Kotor", "Tivat", "Budva", "Bar", "Ulcinj"],
    "CENTRE_Montenegro": ["Nikšic", "Danilovgrad", "Cetinje", "Podgorica"],
    "NORTH_Montenegro": ["Pljevlja", "Plužine", "Žabljak", "Bijelo Polje", "Šavnik", "Mojkovac", "Rožaje", "Berane", "Kolašin", "Andrijevica", "Plav"],
})
GADM_level1_other = GADM_level1_other.merge(Montenegro)

# ============================================================
# North Macedonia
# ============================================================
# Outcome HH7 regions:
#   EAST, NORTHEAST, PELAGONIJA, POLOG, SKOPJE,
#   SOUTHEAST, SOUTHWEST, VARDAR
#
# We add "_North_Macedonia" to HH7 names to avoid generic
# region names such as EAST/SOUTHWEST being confused across countries.
# Manual fixes in R will map the survey HH7 names to these sampled names.
# ============================================================

North_Macedonia = make_country_l1("Macedonia", {
    "EAST_North_Macedonia": [
        "Berovo",
        "Češinovo-Obleševo",
        "Delčevo",
        "Karbinci",
        "Kočani",
        "Makedonska Kamenica",
        "Pehčevo",
        "Probištip",
        "Štip",
        "Vinitsa",
        "Zrnovci"
    ],
    
    "NORTHEAST_North_Macedonia": [
        "Kratovo",
        "Kriva Palanka",
        "Kumanovo",
        "Lipkovo",
        "Rankovce",
        "Staro Nagoričane"
    ],
    
    "PELAGONIJA_North_Macedonia": [
        "Bitola",
        "Demir Hisar",
        "Dolneni",
        "Krivogaštani",
        "Kruševo",
        "Mogila",
        "Novatsi",
        "Prilep",
        "Resen"
    ],
    
    "POLOG_North_Macedonia": [
        "Bogovinje",
        "Brvenica",
        "Gostivar",
        "Jegunovtse",
        "Mavrovo and Rostuša",
        "Tearce",
        "Tetovo",
        "Vrapčište",
        "Želino"
    ],
    
    "SKOPJE_North_Macedonia": [
        "Aerodrom",
        "Aracinovo",
        "Butel",
        "Čair",
        "Centar",
        "Čučer Sandevo",
        "Gazi Baba",
        "Gjorče Petrov",
        "Ilinden",
        "Karpoš",
        "Kisela Voda",
        "Petrovec",
        "Saraj",
        "Sopište",
        "Studeničani",
        "Šuto Orizari",
        "Zelenikovo"
    ],
    
    "SOUTHEAST_North_Macedonia": [
        "Bogdanci",
        "Bosilovo",
        "Star Dojran",
        "Gevgelija",
        "Konče",
        "Novo Selo",
        "Radoviš",
        "Strumitsa",
        "Valandovo",
        "Vasilevo"
    ],
    
    "SOUTHWEST_North_Macedonia": [
        "Centar župa",
        "Debar",
        "Debarca",
        "Kičevo",
        "Zajas",
        "Vraneštica",
        "Oslomej",
        "Drugovo",
        "Makedonski Brod",
        "Ohrid",
        "Plasnica",
        "Struga",
        "Vevčani"
    ],
    
    "VARDAR_North_Macedonia": [
        "Čaška",
        "Demir Kapija",
        "Gradsko",
        "Kavadartsi",
        "Lozovo",
        "Negotino",
        "Rosoman",
        "Sveti Nikole",
        "Veles"
    ],
})

GADM_level1_other = GADM_level1_other.merge(North_Macedonia)

# ============================================================
# Serbia
# ============================================================

Serbia = make_country_l1("Serbia", {
    "BELGRADE_Serbia": ["Grad Beograd"],
    "VOJVODINA_Serbia": ["Severno-Bački", "Severno-Banatski", "Zapadno-Bački", "Srednje-Banatski", "Južno-Bački", "Južno-Banatski", "Sremski"],
    "SUMADIJA_AND_WESTERN_SERBIA_Serbia": ["Mačvanski", "Kolubarski", "Šumadijski", "Pomoravski", "Moravički", "Zlatiborski", "Rasinski", "Raški"],
    "SOUTHERN_AND_EASTERN_SERBIA_Serbia": ["Braničevski", "Podunavski", "Borski", "Zaječarski", "Nišavski", "Toplički", "Pirotski", "Jablanički", "Pčinjski"],
})
GADM_level1_other = GADM_level1_other.merge(Serbia)


# ============================================================
# Thailand
# ============================================================

Thailand = make_country_l1("Thailand", {
    "Bangkok_Thailand": ["Bangkok Metropolis"],
    "Central_Thailand": ["Samut Prakan", "Nonthaburi", "Pathum Thani", "Phra Nakhon Si Ayutthaya", "Ang Thong", "Lop Buri", "Sing Buri", "Chai Nat", "Saraburi", "Chon Buri", "Rayong", "Chanthaburi", "Trat", "Chachoengsao", "Prachin Buri", "Nakhon Nayok", "Sa Kaeo", "Ratchaburi", "Kanchanaburi", "Suphan Buri", "Nakhon Pathom", "Samut Sakhon", "Samut Songkhram", "Phetchaburi", "Prachuap Khiri Khan"],
    "North_Thailand": ["Chiang Mai", "Lamphun", "Lampang", "Uttaradit", "Phrae", "Nan", "Phayao", "Chiang Rai", "Mae Hong Son", "Nakhon Sawan", "Uthai Thani", "Kamphaeng Phet", "Tak", "Sukhothai", "Phitsanulok", "Phichit", "Phetchabun"],
    "Northeast_Thailand": ["Nakhon Ratchasima", "Buri Ram", "Surin", "Si Sa Ket", "Ubon Ratchathani", "Yasothon", "Chaiyaphum", "Amnat Charoen", "Bueng Kan", "Nong Bua Lam Phu", "Khon Kaen", "Udon Thani", "Loei", "Nong Khai", "Maha Sarakham", "Roi Et", "Kalasin", "Sakon Nakhon", "Nakhon Phanom", "Mukdahan"],
    "South_Thailand": ["Nakhon Si Thammarat", "Krabi", "Phangnga", "Phuket", "Surat Thani", "Ranong", "Chumphon", "Songkhla", "Satun", "Trang", "Phatthalung", "Pattani", "Yala", "Narathiwat"],
})
GADM_level1_other = GADM_level1_other.merge(Thailand)


# ============================================================
# Trinidad and Tobago
# ============================================================

Trinidad_Tobago = make_country_l1("Trinidad and Tobago", {
    "North_West_RHA_Trinidad_and_Tobago": ["Arima", "Diego Martin", "Port of Spain", "San Juan-Laventille", "Tunapuna/Piarco"],
    "North_Central_RHA_Trinidad_and_Tobago": ["Chaguanas", "Couva-Tabaquite-Talparo"],
    "Eastern_RHA_Trinidad_and_Tobago": ["Sangre Grande", "Mayaro/Rio Claro"],
    "South_West_RHA_Trinidad_and_Tobago": ["Penal-Debe", "Point Fortin", "Princes Town", "San Fernando", "Siparia"],
    "Tobago_RHA_Trinidad_and_Tobago": ["Tobago"],
})
GADM_level1_other = GADM_level1_other.merge(Trinidad_Tobago)


# ============================================================
# Kyrgyzstan
# ============================================================

Kyrgyzstan = make_country_l1("Kyrgyzstan", {
    "Issyk_kul_Kyrgyzstan": ["Ysyk-Köl"],
    "Jalal_Abad_Kyrgyzstan": ["Jalal-Abad"],
    "Naryn_Kyrgyzstan": ["Naryn"],
    "Batken_Kyrgyzstan": ["Batken"],
    "Osh_Kyrgyzstan": ["Osh"],
    "Talas_Kyrgyzstan": ["Talas"],
    "Chui_Kyrgyzstan": ["Chüy"],
    "Bishkek_c_Kyrgyzstan": ["Biškek"],
    "Osh_c_Kyrgyzstan": ["Osh (city)"],
})
GADM_level1_other = GADM_level1_other.merge(Kyrgyzstan)


# ============================================================
# Uzbekistan
# ============================================================

Uzbekistan = make_country_l1("Uzbekistan", {
    "REPUBLIC_OF_KARAKALPAKSTAN_Uzbekistan": ["Karakalpakstan"],
    "ANDIJAN_REGION_Uzbekistan": ["Andijon"],
    "BUKHARA_REGION_Uzbekistan": ["Bukhoro"],
    "JIZZAKH_REGION_Uzbekistan": ["Jizzakh"],
    "KASHKADARYA_REGION_Uzbekistan": ["Kashkadarya"],
    "NAVOI_REGION_Uzbekistan": ["Navoi"],
    "NAMANGAN_REGION_Uzbekistan": ["Namangan"],
    "SAMARKAND_REGION_Uzbekistan": ["Samarkand"],
    "SURKHANDARYA_REGION_Uzbekistan": ["Surkhandarya"],
    "SYRDARYA_REGION_Uzbekistan": ["Sirdaryo"],
    "TASHKENT_REGION_Uzbekistan": ["Tashkent"],
    "FERGANA_REGION_Uzbekistan": ["Ferghana"],
    "KHOREZM_REGION_Uzbekistan": ["Khorezm"],
    "TASHKENT_CITY_Uzbekistan": ["Tashkent City"],
})
GADM_level1_other = GADM_level1_other.merge(Uzbekistan)


# ============================================================
# Yemen — Hadramawt split to separate Socotra
# ============================================================

Yemen_standard_l1 = make_country_l1("Yemen", {
    "Ibb_Yemen": ["Ibb"],
    "Abyan_Yemen": ["Abyan"],
    "Sanaa_City_Yemen": ["Amanat Al Asimah"],
    "Al_Bayda_Yemen": ["Al Bayda'"],
    "Taizz_Yemen": ["Ta`izz"],
    "Al_Jawf_Yemen": ["Al Jawf"],
    "Hajjah_Yemen": ["Hajjah"],
    "Al_Hudaydah_Yemen": ["Al Hudaydah"],
    "Dhamar_Yemen": ["Dhamar"],
    "Shabwah_Yemen": ["Shabwah"],
    "Saada_Yemen": ["Sa'dah"],
    "Sanaa_Yemen": ["San`a'"],
    "Aden_Yemen": ["`Adan"],
    "Lahj_Yemen": ["Lahij"],
    "Marib_Yemen": ["Ma'rib"],
    "Al_Mahwit_Yemen": ["Al Mahwit"],
    "Al_Maharah_Yemen": ["Al Mahrah"],
    "Amran_Yemen": ["Amran"],
    "Al_Dhalee_Yemen": ["Al Dali'"],
    "Raymah_Yemen": ["Raymah"],
})

SOCOTRA_Yemen_fc = (
    GADM_level2_prep
    .filterMetadata("NAME_0", "equals", "Yemen")
    .filterMetadata("NAME_1", "equals", "Hadramawt")
    .filter(ee.Filter.inList("NAME_2", [
        "Qulensya Wa Abd Al Kuri",
        "Hidaybu"
    ]))
)

HADRAMAWT_Yemen_fc = (
    GADM_level1_prep
    .filterMetadata("NAME_0", "equals", "Yemen")
    .filterMetadata("NAME_1", "equals", "Hadramawt")
)

Hadramaut_Yemen = ee.FeatureCollection([
    ee.Feature(
        HADRAMAWT_Yemen_fc.geometry().difference(SOCOTRA_Yemen_fc.geometry(), 10)
    )
    .set("NAME_0", "Yemen")
    .set("NAME_1", "Hadramaut_Yemen")
    .set("HH7", "Hadramaut_Yemen")
    .set("NAME_1s", HADRAMAWT_Yemen_fc.aggregate_array("NAME_1").join("|"))
    .set("GID_1s", HADRAMAWT_Yemen_fc.aggregate_array("GID_1").join("|"))
])

Socotra_Yemen = ee.FeatureCollection([
    make_feature(
        SOCOTRA_Yemen_fc,
        "Yemen",
        "Socotra_Yemen",
        "Socotra_Yemen",
        level=2
    )
])

Yemen = Yemen_standard_l1.merge(Hadramaut_Yemen).merge(Socotra_Yemen)

GADM_level1_other = GADM_level1_other.merge(Yemen)

# ============================================================
# Exclusions from GADM-based construction
# ============================================================

print("Excluded Kazakhstan: CSV recommends HDX because GADM is erroneous in the east.")
print("Excluded Pakistan Sindh: CSV recommends HDX.")
print("Excluded Yemen Socotra: CSV says boundary must be drawn manually.")
print("Excluded Nauru LOCATION: CSV says likely MICS error.")

# ============================================================
# Exclude GADM_other countries where MICS_other boundaries exist
# ============================================================

mics_countries_raw = set(mics_other_gdf["CNTRYNAME"].dropna().unique())

mics_countries_for_gadm = {
    mics_country_name_map.get(country, country)
    for country in mics_countries_raw
}

print("\nMICS_other countries excluded from GADM_level1_other:")
for country in sorted(mics_countries_for_gadm):
    print(country)

GADM_level1_other = GADM_level1_other.filter(
    ee.Filter.inList(
        "NAME_0",
        list(mics_countries_for_gadm)
    ).Not()
)

# ============================================================
# Summary check
# ============================================================

other_summary = GADM_level1_other.aggregate_histogram("NAME_0").getInfo()

print("\nOther MICS regions by country:")
for country, n in sorted(other_summary.items()):
    print(f"{country}: {n}")

print("\nTotal countries:", len(other_summary))
print("Total regions:", sum(other_summary.values()))



Excluded Kazakhstan: CSV recommends HDX because GADM is erroneous in the east.
Excluded Pakistan Sindh: CSV recommends HDX.
Excluded Yemen Socotra: CSV says boundary must be drawn manually.
Excluded Nauru LOCATION: CSV says likely MICS error.

MICS_other countries excluded from GADM_level1_other:
Comoros
Nigeria
Thailand

Other MICS regions by country:
Afghanistan: 34
Argentina: 7
Belarus: 7
Costa Rica: 7
Cuba: 16
Jamaica: 14
Kyrgyzstan: 9
Macedonia: 8
Montenegro: 3
Nauru: 14
Serbia: 4
Trinidad and Tobago: 5
Turkmenistan: 6
Uzbekistan: 14
Yemen: 22

Total countries: 15
Total regions: 170


In [53]:
analysisYearByCountry_other = {
    'Afghanistan': 2022,
    'Argentina': 2019,
    'Belarus': 2019,
    'Costa Rica': 2018,
    'Cuba': 2019,
    'Jamaica': 2022,
    'Kyrgyzstan': 2023,
    'Montenegro': 2018,
    'Nauru': 2023,
    'Macedonia': 2018,
    'Serbia': 2019,
    'Trinidad and Tobago': 2022,
    'Turkmenistan': 2024,
    'Uzbekistan': 2021,
    'Yemen': 2022
}


### 1c. Derive Analysis Years from Loaded Data

Builds country→year lookup dicts from the loaded GeoDataFrames and applies any boundary-year overrides.

In [54]:
# For DHS survey we can use the year "SVYYEAR" and for MICS "SVYYEARS" which define the start year of the survey collection.
# Keep source-specific country keys so the same country can appear more than once.

dhs_country_years = dhs_gdf.groupby('CNTRYNAMEE')['SVYYEAR'].first().apply(int).to_dict()
mics_country_years = mics_gdf.groupby('CNTRYNAME')['SVYYEARS'].first().apply(int).to_dict()
other_mics_country_years = mics_other_gdf.groupby('CNTRYNAME')['SVYYEARS'].first().apply(int).to_dict()
other_country_years = (
    other_gdf
    .groupby("country")["analysis_year"]
    .first()
    .apply(int)
    .to_dict()
)

analysisYearByCountrySource = {}

for country, year in dhs_country_years.items():
    analysisYearByCountrySource[f"{country}_DHS"] = year

for country, year in mics_country_years.items():
    analysisYearByCountrySource[f"{country}_MICS"] = year

for country, year in other_mics_country_years.items():
    analysisYearByCountrySource[f"{country}_MICS_other"] = year

for country, year in other_country_years.items():
    analysisYearByCountrySource[f"{country}_other"] = year

sortedAnalysisYearByCountrySource = dict(sorted(analysisYearByCountrySource.items()))

for country_source, year in sortedAnalysisYearByCountrySource.items():
    print(f"{country_source}: {year}")

Afghanistan_DHS: 2015
Albania_DHS: 2017
Angola_DHS: 2023
Armenia_DHS: 2016
Bangladesh_DHS: 2022
Benin_DHS: 2017
Bolivia_DHS: 2008
Burkina Faso_DHS: 2021
Burundi_DHS: 2016
Cambodia_DHS: 2021
Cameroon_DHS: 2022
Chad_DHS: 2014
Colombia_DHS: 2015
Comoros_MICS_other: 2022
Congo Democratic Republic_DHS: 2023
Cote d'Ivoire_DHS: 2021
Eswatini_MICS: 2021
Ethiopia_DHS: 2019
Gabon_DHS: 2019
Gambia_MICS: 2018
Georgia_MICS: 2018
Ghana_DHS: 2022
Ghana_MICS: 2017
Guatemala_DHS: 2015
Guinea_DHS: 2021
Haiti_DHS: 2016
Honduras_MICS: 2019
India_DHS: 2020
Indonesia_DHS: 2017
Jordan_DHS: 2023
Kazakhstan_other: 2024
Kenya_DHS: 2022
Kosovo_MICS: 2019
Lao People's Democratic Republic_MICS: 2017
Lesotho_DHS: 2023
Lesotho_MICS: 2018
Liberia_DHS: 2019
Madagascar_DHS: 2021
Madagascar_MICS: 2018
Malawi_DHS: 2024
Malawi_MICS: 2019
Maldives_DHS: 2016
Mali_DHS: 2023
Mauritania_DHS: 2020
Mozambique_DHS: 2022
Myanmar_DHS: 2016
Nepal_DHS: 2022
Nepal_other: 2024
Niger_DHS: 2021
Nigeria_DHS: 2024
Nigeria_MICS: 2016
Nigeri

In [55]:
# Override analysis year where boundary file year differs from sampling year
# These are intentional: use available boundaries, but sample covariates for the listed survey/water-quality year.

boundary_year_overrides = {
    ("Lao People's Democratic Republic", "MICS"): 2023
}

# Apply to MICS
for (country, source), year in boundary_year_overrides.items():
    if source == "MICS":
        mask = mics_gdf["CNTRYNAME"].eq(country)
        if mask.any():
            mics_gdf.loc[mask, "analysis_year"] = year
        else:
            print(f"Warning: MICS override country not found: {country}")

# Use analysis_year rather than SVYYEARS for sampling

## 2. Validation Against Overview

Checks that every row in `overview_water_quality_data_v2.csv` has a matching boundary in the loaded shapefiles.

In [56]:
# Validate overview_water_quality_data_v2.csv against loaded DHS, MICS, other, and GADM boundaries

overview_path = "data/overview_water_quality_data_v2.csv"
overview = pd.read_csv(overview_path, encoding="utf-8-sig")

# Clean column names and values
overview.columns = overview.columns.str.strip()
overview["Country"] = overview["Country"].astype(str).str.strip()
overview["Boundaries"] = overview["Boundaries"].astype(str).str.strip()
overview["Year"] = overview["Year"].astype(int)

# Build available country-year-source records from loaded boundary files
available_records = []

# DHS
for country, years in dhs_gdf.groupby("CNTRYNAMEE")["SVYYEAR"]:
    for year in sorted(pd.Series(years).dropna().astype(int).unique()):
        available_records.append({
            "Country": country,
            "Year": year,
            "Boundaries": "DHS"
        })

# MICS
for country, years in mics_gdf.groupby("CNTRYNAME")["analysis_year"]:
    for year in sorted(pd.Series(years).dropna().astype(int).unique()):
        available_records.append({
            "Country": country,
            "Year": year,
            "Boundaries": "MICS"
        })

# other
if "country" in other_gdf.columns and "analysis_year" in other_gdf.columns:
    for country, years in other_gdf.groupby("country")["analysis_year"]:
        for year in sorted(pd.Series(years).dropna().astype(int).unique()):
            available_records.append({
                "Country": country,
                "Year": year,
                "Boundaries": "other"
            })

# GADM survey-year boundaries from GADM_toSample
gadm_summary = (
    GADM_toSample
    .aggregate_histogram("NAME_0")
    .getInfo()
)

if "analysisYearByCountry" not in globals():
    raise ValueError("analysisYearByCountry is not defined. It is needed to validate GADM survey years.")

for country in sorted(gadm_summary.keys()):
    if country in analysisYearByCountry:
        available_records.append({
            "Country": country,
            "Year": int(analysisYearByCountry[country]),
            "Boundaries": "GADM"
        })
    else:
        available_records.append({
            "Country": country,
            "Year": None,
            "Boundaries": "GADM"
        })

available = pd.DataFrame(available_records)

print("\nAvailable records for other:")
print(
    available[available["Boundaries"] == "other"]
    .drop_duplicates()
    .sort_values(["Country", "Year"])
    .to_string(index=False)
)

# Exact match validation
validation = overview.merge(
    available.assign(found=True),
    on=["Country", "Year", "Boundaries"],
    how="left"
)

validation["found"] = validation["found"].fillna(False)

print("\nOverview rows not found in loaded boundary data:")
missing = validation[~validation["found"]].copy()

if missing.empty:
    print("All overview rows matched country + year + boundary source.")
else:
    print(missing[["Country", "Year", "Boundaries"]].to_string(index=False))

# Helpful diagnostics: source exists but year mismatch
print("\nRows where country + boundary source exists, but year differs:")
country_source_check = overview.merge(
    available,
    on=["Country", "Boundaries"],
    how="left",
    suffixes=("_expected", "_available")
)

year_mismatch = country_source_check[
    country_source_check["Year_available"].notna()
    & (country_source_check["Year_expected"] != country_source_check["Year_available"])
]

if year_mismatch.empty:
    print("No year mismatches found where country + source exists.")
else:
    print(
        year_mismatch[
            ["Country", "Boundaries", "Year_expected", "Year_available"]
        ].drop_duplicates().to_string(index=False)
    )



Available records for other:
   Country  Year Boundaries
Kazakhstan  2024      other
     Nepal  2024      other
  Pakistan  2018      other

Overview rows not found in loaded boundary data:
Country  Year Boundaries
   Fiji  2021       MICS

Rows where country + boundary source exists, but year differs:
No year mismatches found where country + source exists.


<positron-console-cell-56>:84: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


## 3. Shared Definitions

Covariate band list, composite image stack, sampling functions, and the shapefile export helper.
These are used by all three output sections below.

In [57]:
bandNames = [
    'Abs_Lat',
    'CGIAR_Aridity_Index',
    'CGIAR_PET',
    'CHELSA_BIO_Annual_Mean_Temperature',
    'CHELSA_BIO_Annual_Precipitation',
    'CHELSA_BIO_Precipitation_Seasonality',
    'CHELSA_BIO_Precipitation_of_Coldest_Quarter',
    'CHELSA_BIO_Precipitation_of_Driest_Month',
    'CHELSA_BIO_Precipitation_of_Driest_Quarter',
    'CHELSA_BIO_Precipitation_of_Warmest_Quarter',
    'CHELSA_BIO_Precipitation_of_Wettest_Month',
    'CHELSA_BIO_Precipitation_of_Wettest_Quarter',
    'CHELSA_BIO_Temperature_Annual_Range',
    'CHELSA_BIO_Temperature_Seasonality',
    'CIFOR_TropicalPeatlandExtent',
    'CSP_Global_Human_Modification',
    'ConsensusLandCoverClass_Barren',
    'ConsensusLandCoverClass_Cultivated_and_Managed_Vegetation',
    'ConsensusLandCoverClass_Deciduous_Broadleaf_Trees',
    'ConsensusLandCoverClass_Evergreen_Broadleaf_Trees',
    'ConsensusLandCoverClass_Evergreen_Deciduous_Needleleaf_Trees',
    'ConsensusLandCoverClass_Herbaceous_Vegetation',
    'ConsensusLandCoverClass_Mixed_Other_Trees',
    'ConsensusLandCoverClass_Open_Water',
    'ConsensusLandCoverClass_Regularly_Flooded_Vegetation',
    'ConsensusLandCoverClass_Shrubs',
    'ConsensusLandCoverClass_Snow_Ice',
    'ConsensusLandCoverClass_Urban_Builtup',
    'ConsensusLandCover_Human_Development_Percentage',
    'CrowtherLab_Tree_Density',
    'EarthEnvCloudCover_CloudForestPrediction',
    'EarthEnvTexture_CoOfVar_EVI',
    'EarthEnvTexture_Contrast_EVI',
    'EarthEnvTexture_Correlation_EVI',
    'EarthEnvTexture_Dissimilarity_EVI',
    'EarthEnvTexture_Entropy_EVI',
    'EarthEnvTexture_Evenness_EVI',
    'EarthEnvTexture_Homogeneity_EVI',
    'EarthEnvTexture_Maximum_EVI',
    'EarthEnvTexture_Range_EVI',
    'EarthEnvTexture_Shannon_Index',
    'EarthEnvTexture_Simpson_Index',
    'EarthEnvTexture_Std_EVI',
    'EarthEnvTexture_Uniformity_EVI',
    'EarthEnvTexture_Variance_EVI',
    'EarthEnvTopoMed_1stOrderPartialDerivEW',
    'EarthEnvTopoMed_1stOrderPartialDerivNS',
    'EarthEnvTopoMed_Eastness',
    'EarthEnvTopoMed_Elevation',
    'EarthEnvTopoMed_Roughness',
    'EarthEnvTopoMed_Slope',
    'EarthEnvTopoMed_TerrainRuggednessIndex',
    'EarthEnvTopoMed_TopoPositionIndex',
    'EsaCci_BurntAreasProbability',
    'FanEtAl_Depth_to_Water_Table_AnnualMean',
    'FanEtAl_Depth_to_Water_Table_AnnualSD',
    'GHS_Population_Density',
    'GPWv4_Population_Density',
    'GiriEtAl_MangrovesExtent',
    'GLW3_RuminantsDistribution_downsampled10km',
    'MODIS_EVI',
    'MODIS_NDVI',
    'MODIS_NPP',
    'PelletierEtAl_SoilAndSedimentaryDepositThicknesses',
    'SG_Absolute_depth_to_bedrock',
    'SG_Bulk_density_015cm',
    'SG_Depth_to_bedrock',
    'SG_H2O_Capacity_015cm',
    'SG_Saturated_H2O_Content_015cm',
    'TootchiEtAl_WetlandsRegularlyFlooded',
    'WCS_Human_Footprint_2009']

In [58]:
# Stack of environmental covariates to sample 
composite = ee.Image.cat([
        ee.Image("projects/crowtherlab/Composite/CrowtherLab_bioComposite_30ArcSec"),
        ee.Image("projects/crowtherlab/Composite/CrowtherLab_climateComposite_30ArcSec"),
        ee.Image("projects/crowtherlab/Composite/CrowtherLab_geoComposite_30ArcSec"),
        ee.Image("projects/crowtherlab/Composite/CrowtherLab_processComposite_30ArcSec"),
        ])



crs = composite.projection().getInfo()['crs']
transform = composite.projection().getInfo()['transform']
print(f"CRS: {crs}")
print(f"Transform: {transform}")

# DYNAMIC COMPOSITE
# Dynamic covariates to sample based on survey year
WorldPop = ee.ImageCollection("projects/sat-io/open-datasets/WORLDPOP/pop")
VIIRS_V21 = ee.ImageCollection("NOAA/VIIRS/DNB/ANNUAL_V21")
VIIRS_V22 = ee.ImageCollection("NOAA/VIIRS/DNB/ANNUAL_V22")
VIIRS_Annual = VIIRS_V21.merge(VIIRS_V22)
CHIRPS_Daily_Precipitation= ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
MODIS_NDVI_EVI = ee.ImageCollection('MODIS/061/MOD13Q1')
JRC_Built_Up = ee.ImageCollection("JRC/GHSL/P2023A/GHS_BUILT_S")
JRC_DegreeUrban = ee.ImageCollection("JRC/GHSL/P2023A/GHS_SMOD_V2-0")
JRC_Building_Height = ee.ImageCollection("JRC/GHSL/P2023A/GHS_BUILT_H")
JRC_Population = ee.ImageCollection("JRC/GHSL/P2023A/GHS_POP")
ERA5_Monthly_Temp = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")

# STATIC COMPOSITE
MAP_FrictionSurface = ee.Image("projects/malariaatlasproject/assets/accessibility/friction_surface/2019_v5_1")
staticComposite = composite.select(bandNames).addBands(MAP_FrictionSurface.select('friction').rename('map_friction'))

# rename JRC building height image
JRC_Building_Height = ee.Image("JRC/GHSL/P2023A/GHS_BUILT_H/2018") \
    .select('built_height') \
    .rename('jrc_building_height')

# Add it as a new band
staticComposite = staticComposite.addBands(JRC_Building_Height)

CRS: EPSG:4326
Transform: [0.008333333333333333, 0, -180, 0, -0.008333333333333333, 88]


In [59]:
# Sample static and dynamic covariates for a single feature
def sampleFeature(feature):
    f = ee.Feature(feature)
    geom = f.geometry()
    reducer = ee.Reducer.mean()

    static_dict = staticComposite.reduceRegion(
        reducer=reducer,
        crs=crs,
        crsTransform=transform,
        geometry=geom,
        maxPixels=1e13,
        tileScale=16
    )

    dynamic_dict = getDynamicImage(ee.Number(f.get('analysis_year'))).reduceRegion(
        reducer=reducer,
        crs=crs,
        crsTransform=transform,
        geometry=geom,
        maxPixels=1e13,
        tileScale=16
    )


    return f.set(static_dict).set(dynamic_dict)

# GHSL built surface: linearly interpolate between 5-year intervals
# 2015-2019 -> between 2015 and 2020
# 2020-2024 -> between 2020 and 2025
def getGHSLBuiltInterpolated(year):
    year = ee.Number(year)

    y0 = year.divide(5).floor().multiply(5).max(2015).min(2020)
    y1 = y0.add(5)
    w = year.subtract(y0).divide(5)

    img0 = ee.Image(
        JRC_Built_Up.filterDate(ee.Date.fromYMD(y0, 1, 1), ee.Date.fromYMD(y0.add(1), 1, 1)).first()
    ).select('built_surface')

    img1 = ee.Image(
        JRC_Built_Up.filterDate(ee.Date.fromYMD(y1, 1, 1), ee.Date.fromYMD(y1.add(1), 1, 1)).first()
    ).select('built_surface')

    return img0.add(img1.subtract(img0).multiply(w)).rename('ghsl_built_surface')

# GHSL population: linearly interpolate between 5-year intervals
def getGHSLPopulationInterpolated(year):
    year = ee.Number(year)

    y0 = year.divide(5).floor().multiply(5).max(1975).min(2025)
    y1 = y0.add(5).min(2030)

    w = ee.Number(
        ee.Algorithms.If(y1.eq(y0), 0, year.subtract(y0).divide(y1.subtract(y0)))
    )

    img0 = ee.Image(
        JRC_Population.filterDate(
            ee.Date.fromYMD(y0, 1, 1),
            ee.Date.fromYMD(y0.add(1), 1, 1)
        ).first()
    ).select('population_count')

    img1 = ee.Image(
        JRC_Population.filterDate(
            ee.Date.fromYMD(y1, 1, 1),
            ee.Date.fromYMD(y1.add(1), 1, 1)
        ).first()
    ).select('population_count')

    return img0.add(img1.subtract(img0).multiply(w)).rename('ghsl_population')

# GHSL Degree of Urbanisation:
def getGHSLUrbanFraction(year):
    year = ee.Number(year)

    y0 = year.divide(5).floor().multiply(5).max(2015).min(2020)

    smod = ee.Image(
        JRC_DegreeUrban.filterDate(ee.Date.fromYMD(y0, 1, 1), ee.Date.fromYMD(y0.add(1), 1, 1)).first()
    ).select('smod_code')

    return smod.gte(21).rename('ghsl_urban_frac') # classifies 21, 22, 23, and 30 as urban and everything below 21 as non-urban
    

def getWorldPopImage(year):
    year = ee.Number(year).int()

    wp = (WorldPop
          .filter(ee.Filter.calendarRange(year, year, 'year'))
          .select('population')
          .mosaic()
          .rename('worldpop'))

    return wp
    
    # Annual dynamic composite image 
def getDynamicImage(year):
    year = ee.Number(year).int()

    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, 'year')

    # Returns a fully masked image with given bands — used as fallback when a collection is empty
    def masked_fallback(bands):
        if isinstance(bands, str):
            bands = [bands]
        return ee.Image.constant([0] * len(bands)).rename(bands).updateMask(ee.Image.constant(0))

    # WorldPop: annual population image mosaicked across all country tiles
    worldpop = getWorldPopImage(year)

    # VIIRS annual average radiance (starts 2012; pre-2012 surveys return NaN)
    viirs = ee.Image(
        VIIRS_Annual.filterDate(start, end)
        .merge(ee.ImageCollection([masked_fallback(['average'])]))
        .first()
    ).select('average').rename('viirs_average')

    # CHIRPS annual precipitation sum
    chirps_annual_precipitation = CHIRPS_Daily_Precipitation \
        .filterDate(start, end) \
        .merge(ee.ImageCollection([masked_fallback(['precipitation'])])) \
        .select('precipitation') \
        .sum() \
        .rename('chirps_annual_precipitation')

    # MODIS annual mean NDVI and EVI
    modis = MODIS_NDVI_EVI \
        .filterDate(start, end) \
        .merge(ee.ImageCollection([masked_fallback(['NDVI', 'EVI'])])) \
        .select(['NDVI', 'EVI']) \
        .mean() \
        .rename(['modis_ndvi', 'modis_evi'])

    # GHSL built-up surface and population (interpolated)
    ghslBuilt = getGHSLBuiltInterpolated(year)

    ghslPopulation = getGHSLPopulationInterpolated(year)

    # GHSL urban fraction
    ghslUrban = getGHSLUrbanFraction(year)

    # ERA5: merge fallback once, reuse for all bands
    era5_with_fallback = ERA5_Monthly_Temp \
        .filterDate(start, end) \
        .merge(ee.ImageCollection([masked_fallback(['temperature_2m', 'runoff_max', 'runoff_min', 'temperature_2m_max'])]))

    era5_temperature_2m = era5_with_fallback \
        .select('temperature_2m').mean().rename('era5_temperature_2m')

    runoff_max_annualmax = era5_with_fallback \
        .select('runoff_max').max().rename('runoff_max_annualmax')

    runoff_min_annualmin = era5_with_fallback \
        .select('runoff_min').min().rename('runoff_min_annualmin')

    temperature_2m_max_annualmax = era5_with_fallback \
        .select('temperature_2m_max').max().rename('temperature_2m_max_annualmax')

    return ee.Image.cat([
        worldpop,
        viirs,
        chirps_annual_precipitation,
        modis,
        ghslBuilt,
        ghslPopulation,
        ghslUrban,
        era5_temperature_2m,
        runoff_max_annualmax,
        runoff_min_annualmin,
        temperature_2m_max_annualmax
    ])



In [60]:
def samplePopulationSumsForYear(fc, year):
    year = ee.Number(year).int()

    fc_year = fc.filter(ee.Filter.eq("analysis_year", year))

    wp = getWorldPopImage(year)
    ghsl_pop = getGHSLPopulationInterpolated(year)
    ghsl_pop_proj = ghsl_pop.projection()

    def sample_pop(feature):
        f = ee.Feature(feature)
        geom = f.geometry()

        ghsl_pop_sum_dict = ghsl_pop.rename("ghsl_population_sum").reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=geom,
            crs=ghsl_pop_proj,
            scale=ghsl_pop_proj.nominalScale(),
            maxPixels=1e13,
            tileScale=16
        )

        worldpop_sum_dict = wp.rename("worldpop_sum").reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=geom,
            scale=100,
            maxPixels=1e13,
            tileScale=16
        )

        return f.set(ghsl_pop_sum_dict).set(worldpop_sum_dict)

    return fc_year.map(sample_pop)

In [61]:
def prepare_for_export(gdf):
    """Prepare a GeoDataFrame for shapefile export to GEE.

    - Reprojects to EPSG:4326 (required by Earth Engine).
    - Casts GEO_ID to string if present.
    - Shortens field names to <=10 chars for shapefile compatibility:
        analysis_year -> anl_year
        admin_level   -> adm_level
        mics_boundary_file -> mics_file
        dhs_boundary_file  -> dhs_file
        mics_folder        -> mics_fold
    """
    out = gdf.copy()

    if out.crs is not None and out.crs.to_string() != "EPSG:4326":
        out = out.to_crs("EPSG:4326")

    if "GEO_ID" in out.columns:
        out["GEO_ID"] = out["GEO_ID"].astype(str)

    rename_map = {}
    if "analysis_year" in out.columns:
        rename_map["analysis_year"] = "anl_year"
    if "admin_level" in out.columns:
        rename_map["admin_level"] = "adm_level"
    if "mics_boundary_file" in out.columns:
        rename_map["mics_boundary_file"] = "mics_file"
    if "dhs_boundary_file" in out.columns:
        rename_map["dhs_boundary_file"] = "dhs_file"
    if "mics_folder" in out.columns:
        rename_map["mics_folder"] = "mics_fold"

    out = out.rename(columns=rename_map)
    return out


## 4. Output 1 — Water Quality Survey Boundaries and Covariate Sampling

**Outputs:**
- `outputs/02_boundary_files/` — water quality shapefile
- `outputs/03_sampled_features/water_quality_region_areas_km2.csv` — region areas
- GEE Drive exports — static/dynamic covariates and population sums per year

### 4a. Build Water Quality Boundary GeoDataFrame

Filters DHS, MICS, other, and GADM boundaries to those listed in the overview CSV.
Resulting `water_quality_gdf` is used for area calculations and was previously uploaded to GEE as `water_quality_boundaries_tmp_v4`.

In [62]:
os.makedirs("data", exist_ok=True)

# Build one water-quality boundary GeoDataFrame from validated overview list

overview["wq_key"] = (
    overview["Country"] + "_" +
    overview["Year"].astype(str) + "_" +
    overview["Boundaries"]
)
expected_keys = set(overview["wq_key"])

dhs_wq = dhs_gdf.copy()
dhs_wq["country"] = dhs_wq["CNTRYNAMEE"]
dhs_wq["source"] = "DHS"
dhs_wq["wq_key"] = dhs_wq["country"] + "_" + dhs_wq["analysis_year"].astype(str) + "_DHS"
dhs_wq = dhs_wq[dhs_wq["wq_key"].isin(expected_keys)]

mics_wq = mics_gdf.copy()
mics_wq["country"] = mics_wq["CNTRYNAME"]
mics_wq["source"] = "MICS"
mics_wq["wq_key"] = mics_wq["country"] + "_" + mics_wq["analysis_year"].astype(str) + "_MICS"
mics_wq = mics_wq[mics_wq["wq_key"].isin(expected_keys)]

other_wq = other_gdf.copy()
other_wq["source"] = "other"
other_wq["wq_key"] = other_wq["country"] + "_" + other_wq["analysis_year"].astype(str) + "_other"
other_wq = other_wq[other_wq["wq_key"].isin(expected_keys)]

gadm_wq = geemap.ee_to_gdf(
    GADM_toSample.filter(
        ee.Filter.inList("NAME_0", overview.loc[overview["Boundaries"] == "GADM", "Country"].tolist())
    )
)
gadm_wq["country"] = gadm_wq["NAME_0"]
gadm_wq["source"] = "GADM"
gadm_wq["wq_key"] = gadm_wq["country"] + "_" + gadm_wq["analysis_year"].astype(str) + "_GADM"
gadm_wq = gadm_wq[gadm_wq["wq_key"].isin(expected_keys)]

water_quality_gdf = gpd.GeoDataFrame(
    pd.concat([dhs_wq, mics_wq, other_wq, gadm_wq], ignore_index=True),
    crs="EPSG:4326"
)

water_quality_fc = geemap.geopandas_to_ee(water_quality_gdf, geodesic=False)

water_quality_path = "outputs/02_boundary_files/water_quality_boundaries_tmp_v5_notUsed.shp"
if not os.path.exists(water_quality_path):
    prepare_for_export(water_quality_gdf).to_file(water_quality_path)

print(water_quality_gdf.groupby(["source", "country", "analysis_year"]).size())
print("Total water-quality regions:", len(water_quality_gdf))

source  country                           analysis_year
DHS     Cote d'Ivoire                     2021              14
        Malawi                            2024              32
        Mozambique                        2022              11
GADM    Algeria                           2018               7
        Azerbaijan                        2023              13
        Bangladesh                        2019               8
        Benin                             2021              12
        Central African Republic          2018               7
        Chad                              2019              23
        Democratic Republic of the Congo  2017              26
        Dominican Republic                2019              10
        Guinea-Bissau                     2018               9
        Guyana                            2019              10
        Iraq                              2018              18
        Kiribati                          2018               5

### 4b. Area Calculations — Water Quality Regions

Reprojects to equal-area CRS, computes area in km², writes CSV to `outputs/03_sampled_features/`.

In [63]:
# Calculate area of each water quality region and write to CSV.
# Uses water_quality_gdf built in the previous cell.

wq_area_gdf = water_quality_gdf.copy()
wq_area_gdf = wq_area_gdf.to_crs("EPSG:8857")  # equal-area projection
wq_area_gdf["area_km2"] = wq_area_gdf.geometry.area / 1e6  # m² → km²

_wq_region_candidates = ["MICSGEO", "REGNAME", "HH7", "adm1_name", "adm2_name"]

def _get_wq_region_name(row):
    for col in _wq_region_candidates:
        if col in row.index and pd.notna(row[col]) and str(row[col]).strip() != "":
            return row[col]
    return ""

wq_area_gdf["HH7_region"] = wq_area_gdf.apply(_get_wq_region_name, axis=1)

_wq_area_cols = [c for c in ["country", "source", "analysis_year", "admin_level",
                               "HH7_region", "area_km2"] if c in wq_area_gdf.columns]

os.makedirs("outputs/03_sampled_features", exist_ok=True)
wq_area_path = "outputs/03_sampled_features/water_quality_region_areas_km2.csv"
wq_area_gdf[_wq_area_cols].to_csv(wq_area_path, index=False)

print(f"Water quality area CSV written to: {wq_area_path}")
print(f"Regions with no identifier: {(wq_area_gdf['HH7_region'] == '').sum()}")
print(wq_area_gdf[_wq_area_cols].head())


Water quality area CSV written to: outputs/03_sampled_features/water_quality_region_areas_km2.csv
Regions with no identifier: 0
  country source  analysis_year admin_level     HH7_region     area_km2
0  Malawi    DHS           2024           2          Dedza  4058.088950
1  Malawi    DHS           2024           2           Dowa  3076.600695
2  Malawi    DHS           2024           2        Kasungu  8064.515921
3  Malawi    DHS           2024           2       Lilongwe  5827.202833
4  Malawi    DHS           2024           2  Lilongwe City   402.589477


### 4c. After Manual Upload to GEE: Restore Field Names and Export Cleaned Asset

Loads the uploaded shapefile from GEE, restores shortened field names (`anl_year` → `analysis_year`), and exports a cleaned asset ready for sampling.

In [ ]:
water_quality_fc = ee.FeatureCollection("projects/ee-estheregreenwood/assets/2026_worldbank/water_quality_boundaries_tmp_v4")

# rename anl_year back to analysis_year and admin level
water_quality_fc = water_quality_fc.map(lambda f: f.set('analysis_year', f.get('anl_year'))) ## check how analysis_y is coded in your file

water_quality_fc = water_quality_fc.map(lambda f: f.set('admin_level', f.get('admin_leve')))

# export both to GEE
export_water_quality = ee.batch.Export.table.toAsset(
    collection=water_quality_fc,
    description='water_quality_fc_v4',
    assetId='projects/ee-estheregreenwood/assets/2026_worldbank/water_quality_cleaned_fc_v4'
)
export_water_quality.start()  

### 4d. Sample Static and Dynamic Covariates

Loads the cleaned GEE asset and samples all static + dynamic covariates using `sampleFeature()`. Exports to Google Drive.

In [ ]:
water_quality_cleaned_fc = ee.FeatureCollection(
    "projects/ee-estheregreenwood/assets/2026_worldbank/water_quality_cleaned_fc_v4"
)

# Sample static + dynamic covariates only
water_quality_nonpop_fc = water_quality_cleaned_fc.map(sampleFeature)

export_nonpop = ee.batch.Export.table.toDrive(
    collection=water_quality_nonpop_fc,
    description="water_quality_covariates_nonpop_v4",
    folder="GEE_exports_v4",
    fileNamePrefix="water_quality_covariates_nonpop_v4",
    fileFormat="CSV"
)

export_nonpop.start()
print("Started non-population covariate export.")


### 4e. Sample Population Sums by Year

Runs `samplePopulationSumsForYear()` for each unique analysis year in the cleaned feature collection.

In [ ]:
years = water_quality_cleaned_fc.aggregate_array("analysis_year").distinct().sort().getInfo()

for year in years:
    print(f"Starting population export for {year}")

    pop_fc_year = samplePopulationSumsForYear(water_quality_cleaned_fc, year)

    export_pop = ee.batch.Export.table.toDrive(
        collection=pop_fc_year,
        description=f"water_quality_population_sums_{year}",
        folder="GEE_exports",
        fileNamePrefix=f"water_quality_population_sums_{year}",
        fileFormat="CSV"
    )

    export_pop.start()

    

### 4f. Retry: Population Sums for 2021 (Country-by-Country)

The 2021 year-level export timed out. This cell re-submits one task per country for that year.

In [ ]:
# sampling for 2021 timed out. This code attempts to sample again for country surveys gathered in 2021

year = 2021

fc_2021 = water_quality_cleaned_fc.filter(
    ee.Filter.eq("analysis_year", year)
)

countries_2021 = sorted(
    fc_2021.aggregate_array("country").distinct().getInfo()
)

print(f"Countries with {year} boundaries:")
for country in countries_2021:
    print(country)

for country in countries_2021:
    safe_country = (
        country.replace(" ", "_")
        .replace("/", "_")
        .replace("'", "")
        .replace(",", "")
        .replace(".", "")
        .replace("ô", "o")
        .replace("é", "e")
        .replace("è", "e")
        .replace("í", "i")
        .replace("ã", "a")
        .replace("ç", "c")
    )

    print(f"Starting population export for {country}, {year}")

    fc_country_year = fc_2021.filter(
        ee.Filter.eq("country", country)
    )

    pop_fc_country_year = samplePopulationSumsForYear(
        fc_country_year,
        year
    )

    export_pop = ee.batch.Export.table.toDrive(
        collection=pop_fc_country_year,
        description=f"water_quality_population_sums_{year}_{safe_country}",
        folder="GEE_exports",
        fileNamePrefix=f"water_quality_population_sums_{year}_{safe_country}",
        fileFormat="CSV"
    )

    export_pop.start()

## 5. Output 2 — Other Survey Boundaries and Covariate Sampling

**Outputs:**
- `outputs/02_boundary_files/` — combined shapefile for other (non-water-quality) surveys
- `outputs/03_sampled_features/other_surveys_region_areas_km2.csv` — region areas
- GEE Drive exports — static/dynamic covariates and population sums per year

### 5a. Prepare Boundary GeoDataFrames and Export Shapefiles

Sets `country`, `source`, and `analysis_year` fields for DHS, MICS, other, and GADM GDFs, then writes individual shapefiles to `outputs/02_boundary_files/other_surveys/`.

In [67]:
os.makedirs("outputs/02_boundary_files", exist_ok=True)

# ------------------------------------------------------------
# Prepare GeoDataFrames for overview/export
# ------------------------------------------------------------

dhs_export_gdf = dhs_gdf.copy()
dhs_export_gdf["country"] = dhs_export_gdf["CNTRYNAMEE"]
dhs_export_gdf["source"] = "DHS"

mics_export_gdf = mics_gdf.copy()
mics_export_gdf["country"] = mics_export_gdf["CNTRYNAME"]
mics_export_gdf["source"] = "MICS"

mics_other_export_gdf = mics_other_gdf.copy()
mics_other_export_gdf["country"] = mics_other_export_gdf["CNTRYNAME"]
mics_other_export_gdf["source"] = "MICS_other"

other_export_gdf = other_gdf.copy()
other_export_gdf["source"] = "other"


# GADM_toSample is already an EE FeatureCollection, so convert EE -> GeoDataFrame
GADM_wq_gdf = geemap.ee_to_gdf(GADM_toSample)
GADM_wq_gdf["country"] = GADM_wq_gdf["NAME_0"]
GADM_wq_gdf["source"] = "GADM_wq"

# GADM_level1_other is already an EE FeatureCollection, so convert EE -> GeoDataFrame
GADM_other_gdf = geemap.ee_to_gdf(GADM_level1_other)
GADM_other_gdf["country"] = GADM_other_gdf["NAME_0"]
GADM_other_gdf["source"] = "GADM_other"

#gadm_lmic_export_gdf = gadm_lmic_gdf.copy()
#gadm_lmic_export_gdf["country"] = gadm_lmic_export_gdf["NAME_0"]
#gadm_lmic_export_gdf["source"] = "GADM"

# ------------------------------------------------------------
# Add missing metadata to GADM_wq_gdf and GADM_other_gdf
# ------------------------------------------------------------

# analysis year from manual mapping
GADM_other_gdf["analysis_year"] = GADM_other_gdf["country"].map(analysisYearByCountry_other)
GADM_wq_gdf["analysis_year"] = GADM_wq_gdf["country"].map(analysisYearByCountry)

# infer admin level:
# if a feature has GID_2s, treat it as level 2, otherwise level 1
if "GID_2s" in GADM_other_gdf.columns:
    GADM_other_gdf["admin_level"] = (
        GADM_other_gdf["GID_2s"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .map({True: 2, False: 1})
    )
else:
    GADM_other_gdf["admin_level"] = 1

if "GID_2s" in GADM_wq_gdf.columns:
    GADM_wq_gdf["admin_level"] = (
        GADM_wq_gdf["GID_2s"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .map({True: 2, False: 1})
    )
else:
    GADM_wq_gdf["admin_level"] = 1



### 5b. Select Boundaries to Sample

Keeps only surveys from 2015 onward, retains the most recent survey year per country, and writes one combined shapefile for GEE upload.

In [68]:
# ------------------------------------------------------------
# Select boundaries to sample:
# 1) only surveys from 2015 onward
# 2) most recent survey year per country
# 3) write one combined shapefile
# ------------------------------------------------------------

all_boundary_gdfs = [
    dhs_export_gdf,
    mics_export_gdf,
    mics_other_export_gdf,
    other_export_gdf,
    GADM_other_gdf,
    GADM_wq_gdf,
]

country_name_map_for_selection = {
    "The Gambia": "Gambia",
    "Congo Democratic Republic": "Democratic Republic of the Congo",
    "Macedonia": "North Macedonia",
    "Laos": "Lao People's Democratic Republic",
    "Cote d'Ivoire": "Côte d'Ivoire",
}

all_boundaries_gdf = gpd.GeoDataFrame(
    pd.concat(all_boundary_gdfs, ignore_index=True),
    crs="EPSG:4326"
)

all_boundaries_gdf["analysis_year"] = pd.to_numeric(
    all_boundaries_gdf["analysis_year"],
    errors="coerce"
)

all_boundaries_gdf["country_select"] = (
    all_boundaries_gdf["country"]
    .replace(country_name_map_for_selection)
)

# Keep only surveys from 2015 onward
eligible_boundaries_gdf = all_boundaries_gdf[
    all_boundaries_gdf["analysis_year"] >= 2015
].copy()

# Keep the most recent year per country
latest_year_by_country = eligible_boundaries_gdf.groupby(
    "country_select"
)["analysis_year"].transform("max")

selected_boundaries_gdf = eligible_boundaries_gdf[
    eligible_boundaries_gdf["analysis_year"] == latest_year_by_country
].copy()

# Force Pakistan to use DHS rather than most recent survey
pakistan_dhs = eligible_boundaries_gdf[
    (eligible_boundaries_gdf["country_select"] == "Pakistan") &
    (eligible_boundaries_gdf["source"] == "DHS")
].copy()

selected_boundaries_gdf = selected_boundaries_gdf[
    selected_boundaries_gdf["country_select"] != "Pakistan"
]

selected_boundaries_gdf = gpd.GeoDataFrame(
    pd.concat([selected_boundaries_gdf, pakistan_dhs], ignore_index=True),
    crs="EPSG:4326"
)

# Use harmonised country names in final export
selected_boundaries_gdf["country"] = selected_boundaries_gdf["country_select"]
selected_boundaries_gdf = selected_boundaries_gdf.drop(columns="country_select")


# ------------------------------------------------------------
# Overview of selected boundaries
# ------------------------------------------------------------

sampling_overview = (
    selected_boundaries_gdf
    .groupby(["source", "country", "analysis_year", "admin_level"], dropna=False)
    .size()
    .reset_index(name="n_regions")
    .sort_values(["country", "source", "admin_level"])
)

print("\nBoundary files selected for sampling:")
print(sampling_overview.to_string(index=False))

sampling_overview.to_csv(
    "outputs/02_boundary_files/boundary_sampling_overview_selected_v2.csv",
    index=False
)


# ------------------------------------------------------------
# Write one combined shapefile
# ------------------------------------------------------------

combined_out_path = (
    "outputs/02_boundary_files/"
    "all_selected_boundaries_for_sampling_v2.shp"
)

# Remove old combined shapefile parts
for ext in [".shp", ".shx", ".dbf", ".prj", ".cpg"]:
    p = Path(combined_out_path).with_suffix(ext)
    if p.exists():
        p.unlink()

combined_export_gdf = prepare_for_export(selected_boundaries_gdf)

combined_export_gdf.to_file(combined_out_path)

print("\nCombined shapefile export complete:")
print(combined_out_path)
print("Total selected countries:", selected_boundaries_gdf["country"].nunique())
print("Total selected regions:", len(selected_boundaries_gdf))

print("\nShapefile export complete.")
print("Manual step: upload required shapefiles to GEE.")


Boundary files selected for sampling:
    source                          country  analysis_year      admin_level  n_regions
GADM_other                      Afghanistan           2022                1         34
       DHS                          Albania           2017                1         12
   GADM_wq                          Algeria           2018                1          7
       DHS                           Angola           2023                1         18
GADM_other                        Argentina           2019                1          7
       DHS                          Armenia           2016                1         11
   GADM_wq                       Azerbaijan           2023                1         13
       DHS                       Bangladesh           2022                1         16
GADM_other                          Belarus           2019                1          6
GADM_other                          Belarus           2019                2          1
   G

/Users/esthergreenwood/Documents/GitHub/mapping_sanitation_access_LMICs/.venv/lib/python3.9/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field valid_on create as date field, though DateTime requested.
  ogr_write(
/Users/esthergreenwood/Documents/GitHub/mapping_sanitation_access_LMICs/.venv/lib/python3.9/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field valid_to create as date field, though DateTime requested.
  ogr_write(
/Users/esthergreenwood/Documents/GitHub/mapping_sanitation_access_LMICs/.venv/lib/python3.9/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'ADM0_PCODE' to 'ADM0_PCO_1'
  ogr_write(
/Users/esthergreenwood/Documents/GitHub/mapping_sanitation_access_LMICs/.venv/lib/python3.9/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'ADM1_PCODE' to 'ADM1_PCO_1'
  ogr_write(



Combined shapefile export complete:
outputs/02_boundary_files/all_selected_boundaries_for_sampling_v2.shp
Total selected countries: 92
Total selected regions: 1245

Shapefile export complete.
Manual step: upload required shapefiles to GEE.


### 5c. Area Calculations — Other Survey Regions

Reprojects to equal-area CRS, computes area in km², writes CSV to `outputs/03_sampled_features/`.

In [69]:
# ------------------------------------------------------------
# Calculate area of each region and write to CSV
# ------------------------------------------------------------

region_candidates_non_pakistan = [
    "MICSGEO", "REGNAME", "HH7", "adm1_name", "adm2_name"
]
region_candidates_pakistan = [
    "MICSGEO", "REGNAME", "adm2_name", "HH7", "adm1_name"
]

def get_region_name(row):
    candidates = (
        region_candidates_pakistan
        if row.get("country") == "Pakistan"
        else region_candidates_non_pakistan
    )
    for col in candidates:
        if col in row.index and pd.notna(row[col]) and str(row[col]).strip() != "":
            return row[col]
    return ""

area_gdf = selected_boundaries_gdf.copy()
area_gdf["HH7_region"] = area_gdf.apply(get_region_name, axis=1)

# Reproject to an equal-area CRS for accurate area calculation
area_gdf = area_gdf.to_crs("EPSG:8857")  # Mollweide equal-area projection

area_gdf["area_km2"] = area_gdf.geometry.area / 1e6  # m² → km²

area_cols = ["country", "source", "analysis_year", "admin_level", "HH7_region", "area_km2"]

area_csv = area_gdf[area_cols].copy()

area_csv_path = (
    "outputs/03_sampled_features/other_surveys_region_areas_km2.csv"
)

area_csv.to_csv(area_csv_path, index=False)

print(f"\nRegion area CSV written to: {area_csv_path}")
print(f"Regions with no identifier: {(area_csv['HH7_region'] == '').sum()}")
print(area_csv.head())


Region area CSV written to: outputs/03_sampled_features/other_surveys_region_areas_km2.csv
Regions with no identifier: 0
  country source  analysis_year admin_level HH7_region       area_km2
0    Mali    DHS           2023           1     Bamako     245.346332
1    Mali    DHS           2023           1        Gao  178122.818562
2    Mali    DHS           2023           1      Kayes  122488.675367
3    Mali    DHS           2023           1      Kidal  145496.223021
4    Mali    DHS           2023           1  Koulikoro   91002.837096


### 5d. After Manual Upload to GEE: Restore Field Names and Standardise

Loads the uploaded combined shapefile from GEE, restores shortened field names, and standardises region identifier fields so every feature has consistent `HH7`, `NAME_1`, `NAME_2`, etc. properties.

In [ ]:
# ============================================================
# AFTER MANUAL UPLOAD TO GEE:
# Load uploaded boundary files, restore field names,
# check overview, identify specific years,
# and start covariate + population sampling exports
# ============================================================


# ------------------------------------------------------------
# 1. Load uploaded assets from GEE
#    Adjust these asset IDs if your upload names differ
# ------------------------------------------------------------

other_surveys_cleaned_fc = ee.FeatureCollection(
    "projects/ee-estheregreenwood/assets/2026_worldbank/all_selected_boundaries_for_sampling_v2"
)


# Optional: only include this if you also uploaded the full GADM LMIC file
# gadm_lmic_cleaned_fc = ee.FeatureCollection(
#     "projects/ee-estheregreenwood/assets/2026_worldbank/other_surveys/gadm_lmic_fc_cleaned_tmp"
# )



# ------------------------------------------------------------
# 2. Restore shapefile-shortened field names expected by
#    the sampling functions
#    anl_year  -> analysis_year
#    adm_level -> admin_level
# ------------------------------------------------------------

def restore_field_names(feat):
    props = feat.propertyNames()

    analysis_year_value = ee.Algorithms.If(
        props.contains("analysis_year"),
        feat.get("analysis_year"),
        ee.Algorithms.If(
            props.contains("anl_year"),
            feat.get("anl_year"),
            None
        )
    )

    admin_level_value = ee.Algorithms.If(
        props.contains("admin_level"),
        feat.get("admin_level"),
        ee.Algorithms.If(
            props.contains("adm_level"),
            feat.get("adm_level"),
            None
        )
    )

    return (
        feat
        .set("analysis_year", analysis_year_value)
        .set("admin_level", admin_level_value)
    )

other_surveys_cleaned_fc = other_surveys_cleaned_fc.map(restore_field_names)


# ------------------------------------------------------------
# 3. Quick checks
# ------------------------------------------------------------

print("Total features to sample:",
      other_surveys_cleaned_fc.size().getInfo())

print("Properties on first feature:")
print(other_surveys_cleaned_fc.first().propertyNames().getInfo())

print("Sources included:")
print(
    other_surveys_cleaned_fc
    .aggregate_array("source")
    .distinct()
    .sort()
    .getInfo()
)

print("Analysis years included:")
print(
    other_surveys_cleaned_fc
    .aggregate_array("analysis_year")
    .distinct()
    .sort()
    .getInfo()
)


# ------------------------------------------------------------
# 4. Overview of uploaded surveys to be sampled
# ------------------------------------------------------------

overview_df = geemap.ee_to_df(
    other_surveys_cleaned_fc.select(["source", "country", "analysis_year", "admin_level"])
)

overview_df = (
    overview_df
    .groupby(["source", "country", "analysis_year", "admin_level"], dropna=False)
    .size()
    .reset_index(name="n_regions")
    .sort_values(["source", "country", "analysis_year", "admin_level"])
)

print("\nBoundary files that will be sampled:")
print(overview_df.to_string(index=False))



 

In [ ]:
# ------------------------------------------------------------
# 5. Standardise region identifier fields before sampling
# ------------------------------------------------------------
# Make sure every feature has the same key columns.
# This avoids HH7 / NAME_1 being absent from the CSV schema.

def get_first_existing_property(feat, candidate_names, default_value=""):
    props = feat.propertyNames()
    value = default_value
    
    for name in reversed(candidate_names):
        value = ee.Algorithms.If(
            props.contains(name),
            feat.get(name),
            value
        )
    
    return value


def standardise_region_fields(feat):
    hh7_value = get_first_existing_property(
        feat,
        [
            "HH7",
            "MICSGEO",
            "DHSREGEN",
            "REGNAME",
            "adm1_name",
            "ADM1_EN",
            "NAME_1"
        ]
    )
    
    name_1_value = get_first_existing_property(
        feat,
        [
            "NAME_1",
            "adm1_name",
            "ADM1_EN",
            "HH7",
            "MICSGEO",
            "DHSREGEN",
            "REGNAME"
        ],
        default_value=hh7_value
    )
    
    name_2_value = get_first_existing_property(
        feat,
        [
            "NAME_2",
            "adm2_name"
        ]
    )
    
    adm1_name_value = get_first_existing_property(
        feat,
        [
            "adm1_name",
            "ADM1_EN",
            "NAME_1",
            "HH7"
        ],
        default_value=name_1_value
    )
    
    adm2_name_value = get_first_existing_property(
        feat,
        [
            "adm2_name",
            "NAME_2"
        ],
        default_value=name_2_value
    )
    
    return (
        feat
        .set("HH7", hh7_value)
        .set("NAME_1", name_1_value)
        .set("NAME_2", name_2_value)
        .set("adm1_name", adm1_name_value)
        .set("adm2_name", adm2_name_value)
    )


other_surveys_cleaned_fc = other_surveys_cleaned_fc.map(
    standardise_region_fields
)

### 5e. Sample Static and Dynamic Covariates

Maps `sampleFeature()` over the cleaned feature collection, preserving region/country identifiers. Exports to Google Drive.

In [ ]:
# ------------------------------------------------------------
# 6. Sample static + dynamic covariates
#    Keep region/country identifiers from the original boundary features
#    Assumes sampleFeature(feature) already exists
# ------------------------------------------------------------

properties_to_keep = [
    "country",
    "source",
    "analysis_year",
    "admin_level",
    "adm_level",
    "HH7",
    "NAME_0",
    "NAME_1",
    "NAME_2",
    "NAME_3",
    "NAME_1s",
    "NAME_2s",
    "GID_1",
    "GID_2",
    "GID_1s",
    "GID_2s",
    "MICSGEO",
    "DHSREGEN",
    "REGNAME",
    "adm1_name",
    "adm2_name",
    "ADM1_EN"
]

def sampleFeature_keep_properties(feature):
    sampled = ee.Feature(sampleFeature(feature))
    
    return sampled.copyProperties(
        source=feature,
        properties=properties_to_keep
    )

other_surveys_nonpop_fc = other_surveys_cleaned_fc.map(
    sampleFeature_keep_properties
)

# Quick check before export
print("\nProperties after sampling:")
print(other_surveys_nonpop_fc.first().propertyNames().getInfo())

print("\nExample GADM_other HH7 values after sampling:")
print(
    other_surveys_nonpop_fc
    .filter(ee.Filter.eq("source", "GADM_other"))
    .aggregate_array("HH7")
    .slice(0, 20)
    .getInfo()
)

export_nonpop = ee.batch.Export.table.toDrive(
    collection=other_surveys_nonpop_fc,
    description="other_surveys_covariates_nonpop_v3",
    folder="GEE_exports_other_v3",
    fileNamePrefix="other_surveys_covariates_nonpop_v3",
    fileFormat="CSV"
)

export_nonpop.start()
print("\nStarted non-population covariate export.")

### 5f. Sample Population Sums by Year

Standardises `analysis_year` to integer, then runs `samplePopulationSumsForYear()` for each unique year. Exports to Google Drive.

In [ ]:

# ------------------------------------------------------------
# 7. Standardise analysis_year to integer before population exports
# ------------------------------------------------------------

def standardise_analysis_year(feat):
    year_int = ee.Number.parse(ee.String(feat.get("analysis_year"))).int()
    return feat.set("analysis_year", year_int)

other_surveys_cleaned_fc = other_surveys_cleaned_fc.map(
    standardise_analysis_year
)

# ------------------------------------------------------------
# 8. Sample population sums by analysis year
#    Keep region/country identifiers from the original boundary features
# ------------------------------------------------------------

population_export_cols = properties_to_keep + [
    "worldpop_sum",
    "ghsl_population_sum"
]

years = (
    other_surveys_cleaned_fc
    .aggregate_array("analysis_year")
    .distinct()
    .sort()
    .getInfo()
)

for year in years:
    print(f"Starting population export for {year}")

    pop_fc_year = samplePopulationSumsForYear(
        other_surveys_cleaned_fc,
        year
    )

    export_pop = ee.batch.Export.table.toDrive(
        collection=pop_fc_year.select(population_export_cols),
        description=f"other_surveys_population_sums_{year}",
        folder="GEE_exports_other_v3",
        fileNamePrefix=f"other_surveys_population_sums_{year}",
        fileFormat="CSV",
        selectors=population_export_cols
    )

    export_pop.start()

### 5g. Retry: Population Sums for Failed Years (Country-by-Country)

Re-submits population exports at the country level for years that timed out.

In [ ]:
# ------------------------------------------------------------
# Retry population sums country-by-country for failed years
# ------------------------------------------------------------

retry_years = [2019]

import re
import unicodedata

def make_gee_safe_name(text, max_length=100):
    # Convert accented characters to ASCII:
    # São Tomé and Príncipe -> Sao Tome and Principe
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    
    # Replace anything not allowed by GEE with "_"
    text = re.sub(r"[^a-zA-Z0-9.,:;_-]+", "_", text)
    
    # Remove repeated / leading / trailing underscores
    text = re.sub(r"_+", "_", text).strip("_")
    
    return text[:max_length]


for year in retry_years:
    fc_year = other_surveys_cleaned_fc.filter(
        ee.Filter.eq("analysis_year", year)
    )

    countries = (
        fc_year
        .aggregate_array("country")
        .distinct()
        .sort()
        .getInfo()
    )

    for country in countries:
        print(f"Starting population export for {country}, {year}")

        fc_country_year = fc_year.filter(
            ee.Filter.eq("country", country)
        )

        pop_fc_country_year = samplePopulationSumsForYear(
            fc_country_year,
            year
        )

        safe_country = make_gee_safe_name(country)

        description = make_gee_safe_name(
            f"other_surveys_population_sums_{year}_{safe_country}",
            max_length=100
        )

        export_pop = ee.batch.Export.table.toDrive(
            collection=pop_fc_country_year.select(population_export_cols),
            description=description,
            folder="GEE_exports_other_v3",
            fileNamePrefix=description,
            fileFormat="CSV",
            selectors=population_export_cols
        )

        export_pop.start()

### 5h. Retry: Argentina Population Sums (Region-by-Region)

Argentina 2019 failed at country level. This cell submits one task per HH7 region.

In [ ]:
# ------------------------------------------------------------
# Retry Argentina: sample population row by row (by HH7 region) for 2019
# ------------------------------------------------------------

year = 2019
country = "Argentina"

fc_argentina = other_surveys_cleaned_fc.filter(
    ee.Filter.And(
        ee.Filter.eq("analysis_year", year),
        ee.Filter.eq("country", country)
    )
)

regions = (
    fc_argentina
    .aggregate_array("HH7")
    .distinct()
    .sort()
    .getInfo()
)

print(f"Found {len(regions)} regions for {country}, {year}: {regions}")

for region in regions:
    print(f"Starting population export for {country} — HH7={region}, {year}")

    fc_region = fc_argentina.filter(
        ee.Filter.eq("HH7", region)
    )

    pop_fc_region = samplePopulationSumsForYear(fc_region, year)

    safe_region = make_gee_safe_name(str(region))

    description = make_gee_safe_name(
        f"other_surveys_population_sums_{year}_Argentina_HH7_{safe_region}",
        max_length=100
    )

    export_pop = ee.batch.Export.table.toDrive(
        collection=pop_fc_region.select(population_export_cols),
        description=description,
        folder="GEE_exports_other_v3",
        fileNamePrefix=description,
        fileFormat="CSV",
        selectors=population_export_cols
    )

    export_pop.start()

print("All Argentina region exports started.")

## 6. Output 3 — Prediction Data (GADM Admin 1, All LMICs, 2024) [IN PROGRESS]

**Outputs:**
- `outputs/02_boundary_files/` — GADM LMIC shapefile for upload
- `outputs/03_sampled_features/gadm_lmic_region_areas_km2.csv` — region areas
- GEE Drive exports — static/dynamic covariates and population sums (2024)

> **Note:** The prediction shapefile has been edited since it was last uploaded to GEE. The cells below follow the same upload → sample workflow as Outputs 1 and 2.

### 6a. Build Prediction Boundary GeoDataFrame and Export Shapefile

Loads `GADM_lmic_shp` (defined in Section 1), writes an export-ready shapefile to `outputs/02_boundary_files/prediction/` for manual upload to GEE.

In [71]:
# gadm_lmic_gdf is loaded in Section 1 with source="GADM", admin_level=1, analysis_year=2024 already set.
# Write an export-ready shapefile for manual upload to GEE.

os.makedirs("outputs/02_boundary_files", exist_ok=True)
pred_shp_path = "outputs/02_boundary_files/gadm_lmic_pred_2024.shp"
prepare_for_export(gadm_lmic_gdf).to_file(pred_shp_path)

print(f"Prediction shapefile written to: {pred_shp_path}")
print(f"Total prediction regions: {len(gadm_lmic_gdf)}")
print(gadm_lmic_gdf[["NAME_0", "NAME_1"]].head())

Prediction shapefile written to: outputs/02_boundary_files/gadm_lmic_pred_2024.shp
Total prediction regions: 2310
        NAME_0      NAME_1
0  Afghanistan  Badakhshan
1  Afghanistan     Badghis
2  Afghanistan     Baghlan
3  Afghanistan       Balkh
4  Afghanistan      Bamyan


### 6b. Area Calculations — GADM LMIC Admin-1 Regions

Reprojects to equal-area CRS, computes area in km², writes CSV to `outputs/03_sampled_features/`.

In [72]:
# Calculate area of each GADM LMIC admin-1 region and write to CSV.

gadm_area_gdf = gadm_lmic_gdf.to_crs("EPSG:8857")  # equal-area projection
gadm_area_gdf["area_km2"] = gadm_area_gdf.geometry.area / 1e6

os.makedirs("outputs/03_sampled_features", exist_ok=True)
gadm_area_path = "outputs/03_sampled_features/gadm_lmic_region_areas_km2.csv"
gadm_area_gdf[["NAME_0", "NAME_1", "area_km2"]].to_csv(gadm_area_path, index=False)

print(f"GADM area CSV written to: {gadm_area_path}")
print(gadm_area_gdf[["NAME_0", "NAME_1", "area_km2"]].head())


GADM area CSV written to: outputs/03_sampled_features/gadm_lmic_region_areas_km2.csv
        NAME_0      NAME_1      area_km2
0  Afghanistan  Badakhshan  43692.174192
1  Afghanistan     Badghis  20589.905174
2  Afghanistan     Baghlan  21120.258676
3  Afghanistan       Balkh  17253.692189
4  Afghanistan      Bamyan  14173.533520


### 6c. After Manual Upload to GEE: Load Asset and Set Analysis Year

> **TODO:** Update the asset ID below to match the uploaded shapefile path.

Restores shortened field name (`anl_year` → `analysis_year`) and sets `analysis_year = 2024` on all features.

In [73]:
# Load the uploaded GADM prediction asset from GEE.
# TODO: update asset ID to match the asset uploaded from outputs/02_boundary_files/prediction/

pred_fc = ee.FeatureCollection(
    "projects/ee-estheregreenwood/assets/2026_worldbank/gadm_lmic_pred_2024"  # UPDATE after upload
)

# Restore shapefile-shortened field name: anl_year -> analysis_year
pred_fc = pred_fc.map(lambda f: f.set("analysis_year", f.get("anl_year")))

# Set analysis_year = 2024 for all features (prediction target year)
pred_fc_2024 = pred_fc.map(lambda f: ee.Feature(f).set("analysis_year", 2024))

print(f"Total prediction features: {pred_fc_2024.size().getInfo()}")


Total prediction features: 2310


### 6d. Sample Static and Dynamic Covariates

Maps `sampleFeature()` over `pred_fc_2024`. Exports to a GEE asset.

> **TODO:** Update the asset ID if the export description above is changed.

In [74]:
# Sample static + dynamic covariates for all prediction regions.
# Uses sampleFeature() defined in Section 3.

pred_nonpop_fc = pred_fc_2024.map(sampleFeature)

export_pred_covariates = ee.batch.Export.table.toDrive(
    collection=pred_nonpop_fc,
    description="gadm_lmic_pred_covariates_2024",
    folder="GEE_exports",
    fileNamePrefix="nonpop_prediction_v2",
    fileFormat="CSV"
)
export_pred_covariates.start()
print("Started prediction covariate export.")
print(export_pred_covariates.status())


Started prediction covariate export.
{'state': 'READY', 'description': 'gadm_lmic_pred_covariates_2024', 'priority': 100, 'creation_timestamp_ms': 1781527286688, 'update_timestamp_ms': 1781527286688, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'MR7SPD7J4NEN5K7VJGOQGSMQ', 'name': 'projects/ee-estheregreenwood/operations/MR7SPD7J4NEN5K7VJGOQGSMQ'}


In [76]:
print(export_pred_covariates.status())

{'state': 'COMPLETED', 'description': 'gadm_lmic_pred_covariates_2024', 'priority': 100, 'creation_timestamp_ms': 1781527286688, 'update_timestamp_ms': 1781537075364, 'start_timestamp_ms': 1781527288673, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/18BBFK5cwWN5m3QtM9QxPyfAs1qHjhQma'], 'attempt': 1, 'batch_eecu_usage_seconds': 78995.3828125, 'id': 'MR7SPD7J4NEN5K7VJGOQGSMQ', 'name': 'projects/ee-estheregreenwood/operations/MR7SPD7J4NEN5K7VJGOQGSMQ'}


### 6e. Sample Population Sums by Country (2024)

Precomputes 2024 WorldPop and GHSL rasters once, then submits one export task per country to avoid timeouts.

In [ ]:
import ee
import re
import time
import unicodedata

# ------------------------------------------------------------
# pred_fc is intentional
# Set analysis_year = 2024 for all features
# ------------------------------------------------------------
pred_fc_2024 = pred_fc.map(lambda f: ee.Feature(f).set('analysis_year', 2024))


# ------------------------------------------------------------
# Helper: WorldPop image for a given year
# ------------------------------------------------------------
def getWorldPopImage(year):
    year = ee.Number(year).int()

    wp = (WorldPop
          .filter(ee.Filter.calendarRange(year, year, 'year'))
          .select('population')
          .mosaic()
          .rename('worldpop'))

    return wp


# ------------------------------------------------------------
# Helper: GHSL population interpolated for a given year
# ------------------------------------------------------------
def getGHSLPopulationInterpolated(year):
    year = ee.Number(year)

    y0 = year.divide(5).floor().multiply(5).max(1975).min(2025)
    y1 = y0.add(5).min(2030)

    w = ee.Number(
        ee.Algorithms.If(y1.eq(y0), 0, year.subtract(y0).divide(y1.subtract(y0)))
    )

    img0 = ee.Image(
        JRC_Population.filterDate(
            ee.Date.fromYMD(y0, 1, 1),
            ee.Date.fromYMD(y0.add(1), 1, 1)
        ).first()
    ).select('population_count')

    img1 = ee.Image(
        JRC_Population.filterDate(
            ee.Date.fromYMD(y1, 1, 1),
            ee.Date.fromYMD(y1.add(1), 1, 1)
        ).first()
    ).select('population_count')

    return img0.add(img1.subtract(img0).multiply(w)).rename('ghsl_population')


# ------------------------------------------------------------
# Precompute the 2024 rasters once
# ------------------------------------------------------------
wp_2024 = getWorldPopImage(2024)
ghsl_pop_2024 = getGHSLPopulationInterpolated(2024)
ghsl_pop_2024_proj = ghsl_pop_2024.projection()


# ------------------------------------------------------------
# Sample only population sums
# ------------------------------------------------------------
def samplePopulationSums2024(feature):
    f = ee.Feature(feature)
    geom = f.geometry()

    ghsl_pop_sum_dict = ghsl_pop_2024.rename('ghsl_population_sum').reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=geom,
        crs=ghsl_pop_2024_proj,
        scale=ghsl_pop_2024_proj.nominalScale(),
        maxPixels=1e13,
        tileScale=16
    )

    worldpop_sum_dict = wp_2024.rename('worldpop_sum').reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=geom,
        scale=100,
        maxPixels=1e13,
        tileScale=16
    )

    return (f
            .set('analysis_year', 2024)
            .set(ghsl_pop_sum_dict)
            .set(worldpop_sum_dict))


# ------------------------------------------------------------
# Helper: safe file/task names
# ------------------------------------------------------------
def slugify(text):
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'[^A-Za-z0-9]+', '_', text).strip('_')
    return text or 'country'


# ------------------------------------------------------------
# Get unique country names from pred_fc
# ------------------------------------------------------------
countries = sorted(pred_fc_2024.aggregate_array('NAME_0').distinct().getInfo())
print(f"Found {len(countries)} countries")
print(countries[:10])


# ------------------------------------------------------------
# Choose which countries to submit
# Set COUNTRY_END = None to submit all remaining countries
# ------------------------------------------------------------
COUNTRY_START = 101
COUNTRY_END = 130

countries_to_run = countries[COUNTRY_START:COUNTRY_END]
print(f"Submitting {len(countries_to_run)} country exports")


# ------------------------------------------------------------
# Optional: quick 5-feature validation before exports
# ------------------------------------------------------------
validation_fc_5 = (
    pred_fc_2024
    .randomColumn('rand', seed=42)
    .sort('rand')
    .limit(5)
)

feat_list = validation_fc_5.toList(5)

for i in range(5):
    feat = ee.Feature(feat_list.get(i))
    out = samplePopulationSums2024(feat).toDictionary([
        'NAME_0',
        'NAME_1',
        'analysis_year',
        'worldpop_sum',
        'ghsl_population_sum'
    ]).getInfo()

    print(f"\nValidation feature {i+1}")
    print(out)


# ------------------------------------------------------------
# Submit one export task per country
# ------------------------------------------------------------
tasks = []

for country in countries_to_run:
    print(f"Preparing export for: {country}")

    country_fc = pred_fc_2024.filter(ee.Filter.eq('NAME_0', country))
    sampled_country_fc = country_fc.map(samplePopulationSums2024)

    safe_country = slugify(country)

    task = ee.batch.Export.table.toDrive(
        collection=sampled_country_fc,
        description=f'pred_fc_pop_sums_2024_{safe_country}',
        folder='earthengine_exports_pop_sums_2024',
        fileNamePrefix=f'pred_fc_pop_sums_2024_{safe_country}',
        fileFormat='CSV',
        selectors=[
            'NAME_0',
            'NAME_1',
            'analysis_year',
            'worldpop_sum',
            'ghsl_population_sum'
        ]
    )

    task.start()
    tasks.append((country, task))
    print(f"Started task for {country}: {task.id}")

    # Small pause so task creation is gentler
    time.sleep(0.2)


# ------------------------------------------------------------
# Print initial task states
# ------------------------------------------------------------
print("\nInitial task statuses:")
for country, task in tasks:
    status = task.status()
    print(country, status.get('state'), status.get('description'))

### 6f. Retry: Population Sums for Specific Countries

Re-submits population exports for countries whose tasks failed or were skipped.

In [ ]:
# Note: slugify() is defined in cell 6e (section above) and must be run first.

# ------------------------------------------------------------
# Countries to rerun
# ------------------------------------------------------------
TARGET_COUNTRIES = [
    'China',
    'Fiji',
    'Kiribati'
]


# ------------------------------------------------------------
# Check which requested countries are present in NAME_0
# ------------------------------------------------------------
available_countries = sorted(pred_fc_2024.aggregate_array('NAME_0').distinct().getInfo())

countries_to_run = [c for c in TARGET_COUNTRIES if c in available_countries]
missing_countries = [c for c in TARGET_COUNTRIES if c not in available_countries]

print(f"Requested countries: {len(TARGET_COUNTRIES)}")
print(f"Found in pred_fc: {len(countries_to_run)}")
print("Countries to run:", countries_to_run)

if missing_countries:
    print("Not found in NAME_0:", missing_countries)



# ------------------------------------------------------------
# Submit one export task per country
# ------------------------------------------------------------
tasks = []

for country in countries_to_run:
    print(f"\nPreparing export for: {country}")

    country_fc = pred_fc_2024.filter(ee.Filter.eq('NAME_0', country))
    sampled_country_fc = country_fc.map(samplePopulationSums2024)

    safe_country = slugify(country)

    task = ee.batch.Export.table.toDrive(
        collection=sampled_country_fc,
        description=f'pred_fc_pop_sums_2024_{safe_country}',
        folder='earthengine_exports_pop_sums_2024',
        fileNamePrefix=f'pred_fc_pop_sums_2024_{safe_country}',
        fileFormat='CSV',
        selectors=[
            'NAME_0',
            'NAME_1',
            'analysis_year',
            'worldpop_sum',
            'ghsl_population_sum'
        ]
    )

    task.start()
    tasks.append((country, task))
    print(f"Started task for {country}: {task.id}")

    time.sleep(0.2)


# ------------------------------------------------------------
# Print initial task states
# ------------------------------------------------------------
print("\nInitial task statuses:")
for country, task in tasks:
    status = task.status()
    print(country, status.get('state'), status.get('description'))

### 6g. Load Sampled Covariate Results

After the GEE export has completed, loads the sampled feature collection and saves locally as CSV.

> **TODO:** Update asset ID if changed in step 6d.

In [ ]:
# After the covariate export has completed in GEE, load and save locally.
# TODO: update asset ID if description above was changed.

pred_sampled_fc = ee.FeatureCollection(
    "projects/ee-estheregreenwood/assets/2026_worldbank/gadm_lmic_pred_covariates_2024"
)
pred_sampled_gdf = geemap.ee_to_gdf(pred_sampled_fc)

os.makedirs("outputs/03_sampled_features", exist_ok=True)
pred_csv_path = "outputs/03_sampled_features/gadm_lmic_pred_sampled_2024.csv"
pred_sampled_gdf.to_csv(pred_csv_path, index=False)

print(f"Saved {len(pred_sampled_gdf)} rows to {pred_csv_path}")
pred_sampled_gdf.head()
